# Estrategia Bitcoin con clasificación (Google Colab)

**Datos:** archivo CSV de Bitstamp en **tu Google Drive** (tú defines la ruta en el notebook).

**Pedagogía:** **una idea por celda** — lee **Objetivo → En palabras simples → qué hace / qué ver → Confirma por escrito** antes de ejecutar código.

**Partes del cuaderno:** A datos · B preparación · C modelos · D Random Forest final · E backtest.


## Cómo leer este notebook (obligatorio)

1. **Aplicación Bitcoin:** en cada minuto tenemos precio → construimos indicadores → definimos si “conviene estar largo” (`signal`) → entrenamos un modelo que **imita/mejora** esa regla → simulamos ganancias con **backtest**.
2. **Modelos ML:** EDA → separar Y/X → train/validation → comparar algoritmos → Grid Search → evaluar → guardar → backtest.
3. **No es lo mismo:** alta **accuracy** ≠ ganar dinero; **signal=1** ≈ “comprado” en esta demo; el **backtest** es donde ves si la señal ML paga (sin comisiones).

**Convención:** `Y` = columna `signal` (0 no largo, 1 largo). `X` = todo lo demás numérico. **Validation** = datos que el modelo no usa para entrenar en el paso final.


## Contenido del cuaderno

* [Configuración de la sesión](#cfg)
* [1. Definición del problema](#0)
* [2. Librerías y datos](#1)
* [3. EDA (breve)](#2)
* [4. Preparación de datos](#3)
* [5. Evaluación y modelos](#4)
* [6. Grid Search](#5)
* [7. Modelo final](#6)
* [8. Backtesting](#7)
* [9. Modo con más filas (opcional)](#8)
* [Google Drive — ruta del CSV](#drive)


<a id='0'></a>
# 1. Definición del problema

## 1.1 Enfoque de la aplicación (Bitcoin / trading)

| Paso | Qué hacemos | Para qué sirve en trading |
|------|-------------|---------------------------|
| 1 | Cargar OHLCV minuto a minuto (Bitstamp) | Tener la serie de precios y liquidez |
| 2 | Limpiar y ordenar en el tiempo | Evitar señales con huecos falsos |
| 3 | Definir **etiqueta** con SMA 10 vs SMA 60 | Traducir “¿estoy alcista de corto plazo?” en 0/1 |
| 4 | Construir **indicadores** (RSI, estocástico, etc.) | Dar al ML pistas de tendencia y momentum |
| 5 | Entrenar clasificadores | Aproximar (y a veces mejorar) la regla SMA |
| 6 | Elegir modelo + hiperparámetros | Balance entre acierto y estabilidad |
| 7 | Evaluar en **validación** (matriz de confusión) | Ver falsas compras vs subidas perdidas |
| 8 | **Backtest** simple | Simular PnL antes de arriesgar capital |

**Señal de referencia (regla SMA):**

- **1 = compra / largo:** SMA₁₀(Close) > SMA₆₀(Close).
- **0 = no largo:** en caso contrario.

El ML **no sustituye** esa idea: aprende una función $f(X_t) \approx \text{signal}_t$ usando muchos indicadores a la vez.

### Historia completa en una frase (léela en voz alta)

> Cargamos minutos de Bitcoin → limpiamos → marcamos cada minuto con **0/1** según medias móviles → calculamos RSI, EMA, etc. → entrenamos un clasificador para predecir ese 0/1 → medimos si acierta y si **ganaría dinero** retrasando la señal un minuto.

### Dos hilos que SIEMPRE van juntos

| | **Hilo trading** | **Hilo ML** |
|---|------------------|-------------|
| **Entrada** | Precio/volumen minuto a minuto | Filas = observaciones, columnas = features |
| **Decisión** | ¿Largo (1) o no (0)? | Clasificación binaria |
| **Éxito en clase** | Entiendes falsas compras y backtest | Entiendes train/val, CV y Grid Search |

## 1.2 Enfoque de modelos (clasificación supervisada)

| Paso ML | Sección del notebook | Idea |
|---------|----------------------|------|
| 1 | EDA | Entender datos antes de modelar |
| 2 | Preparación | $Y$ = signal, $X$ = indicadores |
| 3 | Train / validation split | Medir generalización |
| 4 | Validación cruzada | Comparar algoritmos sin mirar el test |
| 5 | Grid Search | Ajustar hiperparámetros del ganador |
| 6 | Modelo final + métricas | Reportar en hold-out |
| 7 | Persistencia + backtest | Cerrar el ciclo “investigación → decisión” |

**Datos:** sube tu CSV a Drive (p. ej. `BitstampData_sample.csv` o el archivo completo) y apunta `RUTA_CSV` a esa ubicación.


<a id='1'></a>
# 2. Inicio — librerías y datos
<a id='1.1'></a>
## 2.1 Cargar librerías

Usamos `pandas`, `sklearn`, gráficos y (opcional) `MLPClassifier` como red neuronal simple en Colab.


### Paso 2.1a — Detectar Colab

**Objetivo:** Completar «Detectar Colab» y verificar la salida antes de continuar.

**En palabras simples:** Comprueba si el notebook corre en Google Colab.

**Qué hace el código:** Comprueba si el notebook corre en Google Colab.

**Qué debes ver al ejecutar:** Debe imprimir `Colab: True` o `False`.

**Qué implica para el modelo ML:** En local no hace falta `pip`; en Colab instalamos paquetes si faltan.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Debe imprimir `Colab: True` o `False`.


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('Colab:', IN_COLAB)


### Paso 2.1b — Instalar dependencias (solo Colab)

**Objetivo:** Completar «Instalar dependencias (solo Colab)» y verificar la salida antes de continuar.

**En palabras simples:** Instala `seaborn` y `scikit-learn` si estamos en Colab.

**Qué hace el código:** Instala `seaborn` y `scikit-learn` si estamos en Colab.

**Qué debes ver al ejecutar:** Si no estás en Colab, esta celda no instala nada (está bien).

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Si no estás en Colab, esta celda no instala nada (está bien).


In [ ]:
if IN_COLAB:
    !pip -q install seaborn scikit-learn
else:
    print('Entorno local: usa tu venv / conda.')


### Paso 2.1c — NumPy, Pandas y gráficos

**Objetivo:** Completar «NumPy, Pandas y gráficos» y verificar la salida antes de continuar.

**En palabras simples:** Importa herramientas para tablas y visualización. En trading: Aquí manipularemos precios minuto a minuto (OHLCV).

**Qué hace el código:** Importa herramientas para tablas y visualización.

**Qué debes ver al ejecutar:** Sin errores de importación.

**Qué implica para Bitcoin / trading:** Aquí manipularemos precios minuto a minuto (OHLCV).

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Sin errores de importación.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas import read_csv, set_option
from pandas.plotting import scatter_matrix
import seaborn as sns


### Paso 2.1d — Scikit-learn — preparación y validación

**Objetivo:** Completar «Scikit-learn — preparación y validación» y verificar la salida antes de continuar.

**En palabras simples:** Importa partición de datos, CV y escalado.

**Qué hace el código:** Importa partición de datos, CV y escalado.

**Qué debes ver al ejecutar:** Nombres como `train_test_split`, `GridSearchCV`.

**Qué implica para el modelo ML:** Estas funciones preparan partición de datos, CV y escalado.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Nombres como `train_test_split`, `GridSearchCV`.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score, GridSearchCV, validation_curve,
)


### Paso 2.1e — Scikit-learn — modelos

**Objetivo:** Completar «Scikit-learn — modelos» y verificar la salida antes de continuar.

**En palabras simples:** Importa los clasificadores del benchmark.

**Qué hace el código:** Importa los clasificadores del benchmark.

**Qué debes ver al ejecutar:** Lista de clases sin error.

**Qué implica para el modelo ML:** Cada algoritmo es un candidato para aproximar la etiqueta SMA.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Lista de clases sin error.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier


### Paso 2.1f — Métricas y guardado

**Objetivo:** Completar «Métricas y guardado» y verificar la salida antes de continuar.

**En palabras simples:** Importa métricas de clasificación y `pickle` para persistir el modelo. En trading: Precision/recall traducen errores en falsas compras o subidas perdidas.

**Qué hace el código:** Importa métricas de clasificación y `pickle` para persistir el modelo.

**Qué debes ver al ejecutar:** Funciones `accuracy_score`, `confusion_matrix`, etc.

**Qué implica para Bitcoin / trading:** Precision/recall traducen errores en falsas compras o subidas perdidas.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Funciones `accuracy_score`, `confusion_matrix`, etc.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from pickle import dump, load
print('Librerías listas.')


<a id='cfg'></a>
## Configuración de la sesión (ejecutar primero)

- **`MODO_LIGERO = True`:** 30 000 filas, 3 folds CV, 5 modelos en benchmark, grid RF pequeño (Colab más rápido).
- **`MODO_LIGERO = False`:** 100 000 filas, 10 folds, 9 modelos, grid más grande (más pesado).


### Paso 2.2a — Modo de cómputo

**Objetivo:** Completar «Modo de cómputo» y verificar la salida antes de continuar.

**En palabras simples:** Activa modo ligero o completo según tamaño de CSV y potencia de Colab.

**Qué hace el código:** Activa modo ligero o completo según tamaño de CSV y potencia de Colab.

**Qué debes ver al ejecutar:** `MODO_LIGERO` True o False impreso.

**Qué implica para el modelo ML:** Menos filas/folds = entrenamiento más rápido, menos exhaustivo.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: `MODO_LIGERO` True o False impreso.


In [ ]:
MODO_LIGERO = True
print('Modo ligero:', MODO_LIGERO)


### Paso 2.2b — Tamaño de muestra y CV

**Objetivo:** Completar «Tamaño de muestra y CV» y verificar la salida antes de continuar.

**En palabras simples:** Define cuántas filas finales y cuántos folds usar. En trading: Modelar solo el tramo reciente enfatiza el régimen de mercado actual (con trade-offs).

**Qué hace el código:** Define cuántas filas finales y cuántos folds usar.

**Qué debes ver al ejecutar:** Imprime `N_FILAS` y `N_FOLDS`.

**Qué implica para Bitcoin / trading:** Modelar solo el tramo reciente enfatiza el régimen de mercado actual (con trade-offs).

**Qué implica para el modelo ML:** Más folds = estimación más estable pero más lento.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Imprime `N_FILAS` y `N_FOLDS`.


In [ ]:
N_FILAS = 30_000 if MODO_LIGERO else 100_000
N_FOLDS = 3 if MODO_LIGERO else 10
CV_SEED = 7
print('Filas para modelado:', N_FILAS, '| Folds CV:', N_FOLDS, '| Seed:', CV_SEED)


---
## Parte A

**Qué hacemos aquí (sin jerga):** Vamos a **conectar Google Drive**, **cargar** el CSV de Bitstamp y **mirar** si los precios tienen sentido antes de calcular nada.

**Al terminar la parte A, debes poder decir en voz alta:** Dónde está tu archivo en Drive, cuántas filas hay y qué significa cada columna OHLCV.

**Pregunta guía:** ¿Por qué cada fila es un minuto y por qué eso importa para la estrategia?


<a id='drive'></a>
<a id='1.2'></a>
### 2.3 Datos desde Google Drive

**Antes de la clase:** sube tu CSV a Drive (ejemplo de nombre: `BitstampData_sample.csv`).

**Ruta típica en Colab:** `/content/drive/MyDrive/TU_CARPETA/BitstampData_sample.csv`

**Importante:** en la celda **Paso A2** debes **editar** `RUTA_CSV` con la ruta real de tu archivo (copiar desde el explorador de archivos de Drive).


### Paso A1 — Montar Google Drive

**Objetivo:** Tener acceso de lectura a tu carpeta de Drive.

**En palabras simples:** Conectamos Colab con tu Google Drive como si fuera un disco.

**Qué hace el código:** En Colab, monta Drive para leer archivos con ruta `/content/drive/...`.

**Qué debes ver al ejecutar:** Mensaje de autorización (Colab) o aviso de entorno local.

**Confirma por escrito (cuaderno o chat):** ¿Apareció el enlace de permisos de Google y pudiste montar Drive? (sí/no)


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('No estás en Colab: salta el montaje; usa una ruta local en RUTA_CSV.')


### Paso A2 — Definir RUTA_CSV (EDITAR AQUÍ)

**Objetivo:** Apuntar al archivo correcto antes de leer.

**En palabras simples:** Le dices al notebook dónde está tu tabla de precios dentro de Drive.

**Qué hace el código:** Variable de texto con la ruta **completa** al CSV en Drive (o local).

**Qué debes ver al ejecutar:** Imprime la ruta que configuraste.

**Qué implica para Bitcoin / trading:** Este archivo contiene todos los minutos de precio que usará la estrategia.

**Confirma por escrito (cuaderno o chat):** Mi RUTA_CSV es: _________________________________ (copia la misma ruta del código).


In [ ]:
from pathlib import Path

# --- EDITA SOLO ESTA LÍNEA (ruta en Drive o local) ---
RUTA_CSV = '/content/drive/MyDrive/ML_Fin_Econ/BitstampData_sample.csv'

print('RUTA_CSV configurada:')
print(RUTA_CSV)


### Paso A3 — Comprobar que el archivo existe

**Objetivo:** Detectar rutas mal escritas antes de un error críptico.

**En palabras simples:** Comprobamos que el CSV está donde crees que está.

**Qué hace el código:** Verifica que `Path(RUTA_CSV).exists()` sea True.

**Qué debes ver al ejecutar:** Debe imprimir `¿Existe el archivo? True`.

**Confirma por escrito (cuaderno o chat):** Si sale False: corrige mayúsculas, carpetas y extensión `.csv`.


In [ ]:
archivo_datos = Path(RUTA_CSV)
print('¿Existe el archivo?', archivo_datos.exists())
if not archivo_datos.exists():
    raise FileNotFoundError(
        'No se encontró el CSV. Sube el archivo a Drive y corrige RUTA_CSV en el Paso A2.'
    )


### Paso A4 — Leer el CSV en pandas

**Objetivo:** Completar «Leer el CSV en pandas» y verificar la salida antes de continuar.

**En palabras simples:** `read_csv(RUTA_CSV)` carga todo en `dataset`. En trading: Cada fila = un minuto de mercado Bitstamp.

**Qué hace el código:** `read_csv(RUTA_CSV)` carga todo en `dataset`.

**Qué debes ver al ejecutar:** Imprime número de filas y columnas.

**Qué implica para Bitcoin / trading:** Cada fila = un minuto de mercado Bitstamp.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Imprime número de filas y columnas.


In [ ]:
dataset = read_csv(RUTA_CSV)
print('Archivo leído:', archivo_datos)
print('Filas cargadas:', len(dataset))
print('Columnas:', list(dataset.columns))


### Paso a paso — qué es cada columna (Bitcoin)

| Columna | Significado en mercado |
|---------|-------------------------|
| `Open`, `High`, `Low`, `Close` | Precio en ese minuto (vela de 1 min) |
| `Volume_(BTC)` | Cantidad negociada en BTC (liquidez relativa) |
| `Volume_(Currency)` | Volumen en moneda fiat (se elimina más adelante en el repo) |
| `Weighted_Price` | Precio promedio ponderado del minuto |

Cada **fila = un minuto**. La estrategia decide en cada minuto si el régimen es “alcista de corto plazo” (etiqueta) usando medias e indicadores **calculados solo con información hasta ese minuto** (cuidado con *look-ahead* en proyectos propios).


### Paso A4 — Herramientas de visualización en notebook

**Objetivo:** Completar «Herramientas de visualización en notebook» y verificar la salida antes de continuar.

**En palabras simples:** Configura `display` y precisión de pandas.

**Qué hace el código:** Configura `display` y precisión de pandas.

**Qué debes ver al ejecutar:** `Librerías listas` implícito al ejecutar sin error.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: `Librerías listas` implícito al ejecutar sin error.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print
set_option('display.width', 100)
set_option('precision', 3)


### Paso A5 — Dimensiones del dataset

**Objetivo:** Completar «Dimensiones del dataset» y verificar la salida antes de continuar.

**En palabras simples:** Muestra filas × columnas.

**Qué hace el código:** Muestra filas × columnas.

**Qué debes ver al ejecutar:** Tupla `(filas, columnas)` — anota el número de filas de tu CSV.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Tupla `(filas, columnas)` — anota el número de filas de tu CSV.


In [ ]:
print('Shape (filas, columnas):', dataset.shape)


**✓ Celda ejecutada — interpreta así:**

¿Cuántas filas tienes? En clase usaremos solo las últimas `N_FILAS` minutos.

En **ML:** Menos filas = entrenamiento más rápido pero menos historia.


### Paso A6 — Cola de la serie

**Objetivo:** Completar «Cola de la serie» y verificar la salida antes de continuar.

**En palabras simples:** Muestra los últimos 5 minutos cargados. En trading: Verifica que `Close` tenga sentido (no ceros ni saltos absurdos).

**Qué hace el código:** Muestra los últimos 5 minutos cargados.

**Qué debes ver al ejecutar:** Precios y volúmenes recientes; orden temporal (más nuevo abajo).

**Qué implica para Bitcoin / trading:** Verifica que `Close` tenga sentido (no ceros ni saltos absurdos).

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Precios y volúmenes recientes; orden temporal (más nuevo abajo).


In [ ]:
display(dataset.tail(5))


### Paso A7 — Estadísticas descriptivas

**Objetivo:** Completar «Estadísticas descriptivas» y verificar la salida antes de continuar.

**En palabras simples:** Resumen numérico por columna (`describe`). En trading: Rango de `Close` te sitúa en qué época/precio está la muestra.

**Qué hace el código:** Resumen numérico por columna (`describe`).

**Qué debes ver al ejecutar:** Columna `count`: si es menor que filas, hay NaNs.

**Qué implica para Bitcoin / trading:** Rango de `Close` te sitúa en qué época/precio está la muestra.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Columna `count`: si es menor que filas, hay NaNs.


In [ ]:
display(dataset.describe())


**✓ Celda ejecutada — interpreta así:**

Compara `count` con el número de filas. Anota min/max de `Close`.

En **ML:** NaNs o outliers extremos pueden distorsionar indicadores y modelos.

**Conclusión:** Si `count` < filas, hay missing data; la limpieza de la parte B lo tratará.


> **Actividad A1 (3 min):** *¿Por qué no entrenar con toda la serie desde 2012 en Colab free?* (memoria, tiempo, cambio de régimen).


---
## Parte B

**Qué hacemos aquí (sin jerga):** Preparamos la tabla que el ML consumirá: limpiar → **crear la etiqueta signal (SMA)** → calcular indicadores → quitar columnas que harían trampa.

**Al terminar la parte B, debes poder decir en voz alta:** Qué es `signal`, qué entra en **X**, y por qué quitamos las SMA de la etiqueta antes de entrenar.

**Pregunta guía:** Si el modelo viera `short_mavg` en X, ¿estaría aprendiendo o copiando la regla?


<a id='3'></a>
# 4. Preparación de datos
<a id='3.1'></a>
## 4.1 Limpieza de datos


### Paso B1.1 — Detectar valores faltantes

**Objetivo:** Completar «Detectar valores faltantes» y verificar la salida antes de continuar.

**En palabras simples:** Pregunta si existe algún NaN en la tabla. En trading: Huecos en precios generan señales falsas si no se tratan.

**Qué hace el código:** Pregunta si existe algún NaN en la tabla.

**Qué debes ver al ejecutar:** `Null Values = False` es lo ideal antes de indicadores.

**Qué implica para Bitcoin / trading:** Huecos en precios generan señales falsas si no se tratan.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: `Null Values = False` es lo ideal antes de indicadores.


In [ ]:
print('¿Hay NaNs?', dataset.isnull().values.any())
print('NaNs por columna:\n', dataset.isnull().sum())


### Paso B1.2 — Forward fill

**Objetivo:** Completar «Forward fill» y verificar la salida antes de continuar.

**En palabras simples:** Rellena NaNs con el último valor observado (forward fill). En trading: Asume que el último precio sigue vigente en el hueco — razonable en minutos, dudoso en días.

**Qué hace el código:** Rellena NaNs con el último valor observado (forward fill).

**Qué debes ver al ejecutar:** Repite el chequeo mental: menos NaNs tras ejecutar.

**Qué implica para Bitcoin / trading:** Asume que el último precio sigue vigente en el hueco — razonable en minutos, dudoso en días.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Repite el chequeo mental: menos NaNs tras ejecutar.


In [ ]:
dataset[dataset.columns.values] = dataset[dataset.columns.values].ffill()
print('¿NaNs tras ffill?', dataset.isnull().values.any())


### Paso B1.3 — Quitar marca de tiempo

**Objetivo:** Completar «Quitar marca de tiempo» y verificar la salida antes de continuar.

**En palabras simples:** Elimina `Timestamp`; el orden de filas ya indexa el tiempo.

**Qué hace el código:** Elimina `Timestamp`; el orden de filas ya indexa el tiempo.

**Qué debes ver al ejecutar:** Columna `Timestamp` ya no está en `dataset.columns`.

**Qué implica para el modelo ML:** El modelo no usa la fecha cruda; solo features numéricas.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Columna `Timestamp` ya no está en `dataset.columns`.


In [ ]:
dataset = dataset.drop(columns=['Timestamp'])
print('Columnas:', list(dataset.columns))


<a id='3.2'></a>
## 4.2 Datos categóricos

En otros problemas de ML se codifican variables categóricas (one-hot). **En Bitcoin todas las entradas son numéricas** (OHLCV). Si añadieras calendario (día de la semana), ahí codificarías esa variable.


<a id='3.3'></a>
## 4.3 Preparar datos para clasificación

### Paso a paso — construcción de la etiqueta (aplicación)

1. **Elegir ventanas:** 10 min (corto) y 60 min (largo).
2. **Calcular SMA** sobre `Close`.
3. **Comparar:** corto > largo → etiqueta **1** (régimen alcista de corto plazo).
4. **Operativa:** 1 ≈ “estar largo”; 0 ≈ “no largo” (sin short explícito en el caso base).

$$\text{signal}_t = \mathbb{1}\{\text{SMA}_{10}(P)_t > \text{SMA}_{60}(P)_t\}$$

Las SMA usadas para etiquetar se **eliminan** de $X$ antes de entrenar para evitar copiar la regla trivialmente.


### Paso B3.1 — SMA corta (10 min)

**Objetivo:** Completar «SMA corta (10 min)» y verificar la salida antes de continuar.

**En palabras simples:** Calcula media móvil simple de `Close` con ventana 10. En trading: Captura tendencia muy reciente (≈10 minutos).

**Qué hace el código:** Calcula media móvil simple de `Close` con ventana 10.

**Qué debes ver al ejecutar:** `short_mavg` cerca de `Close` pero más suave.

**Qué implica para Bitcoin / trading:** Captura tendencia muy reciente (≈10 minutos).

**Confirma por escrito (cuaderno o chat):** La salida coincide con: `short_mavg` cerca de `Close` pero más suave.


In [ ]:
dataset['short_mavg'] = dataset['Close'].rolling(window=10, min_periods=1, center=False).mean()
dataset[['Close', 'short_mavg']].tail(3)


### Paso B3.2 — SMA larga (60 min)

**Objetivo:** Completar «SMA larga (60 min)» y verificar la salida antes de continuar.

**En palabras simples:** Media móvil de 60 minutos sobre `Close`. En trading: Proxy de tendencia de la última hora.

**Qué hace el código:** Media móvil de 60 minutos sobre `Close`.

**Qué debes ver al ejecutar:** `long_mavg` cambia más lento que `short_mavg`.

**Qué implica para Bitcoin / trading:** Proxy de tendencia de la última hora.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: `long_mavg` cambia más lento que `short_mavg`.


In [ ]:
dataset['long_mavg'] = dataset['Close'].rolling(window=60, min_periods=1, center=False).mean()
dataset[['Close', 'short_mavg', 'long_mavg']].tail(3)


### Paso B3.3 — Crear etiqueta `signal`

**Objetivo:** Tener la variable objetivo Y definida con la regla SMA 10 vs 60.

**En palabras simples:** Traducimos el cruce de medias en un sí/no numérico para que sklearn pueda entrenar.

**Qué hace el código:** Crea la columna `signal`: 1.0 si `short_mavg > long_mavg`, si no 0.0.

**Qué debes ver al ejecutar:** Columna `signal` solo con 0.0 y 1.0; en `tail` ves 0/1 junto a las medias.

**Qué implica para Bitcoin / trading:** **1 = en este minuto la regla SMA dice ‘entorno alcista de corto plazo’** (en la demo = estar largo). **0 = no.**

**Qué implica para el modelo ML:** **Esta columna es Y (la verdad que el modelo intentará predecir).** No confundir con la predicción ŷ (viene mucho más adelante).

**Confirma por escrito (cuaderno o chat):** En tus palabras: signal=1 significa _______________; signal=0 significa _______________.


In [ ]:
dataset['signal'] = np.where(dataset['short_mavg'] > dataset['long_mavg'], 1.0, 0.0)
dataset[['Close', 'short_mavg', 'long_mavg', 'signal']].tail(5)


### Paso B3.4 — Balance de clases

**Objetivo:** Completar «Balance de clases» y verificar la salida antes de continuar.

**En palabras simples:** Cuenta proporción de 0 y 1 en `signal`. En trading: Muchos 1 → mercado alcista en la ventana; pocos 1 → lateral/bajista.

**Qué hace el código:** Cuenta proporción de 0 y 1 en `signal`.

**Qué debes ver al ejecutar:** Porcentajes en `value_counts(normalize=True)`.

**Qué implica para Bitcoin / trading:** Muchos 1 → mercado alcista en la ventana; pocos 1 → lateral/bajista.

**Qué implica para el modelo ML:** Clase muy mayoritaria infla accuracy de un modelo trivial.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Porcentajes en `value_counts(normalize=True)`.


In [ ]:
dataset['signal'].value_counts(normalize=True)


**✓ Celda ejecutada — interpreta así:**

Si ~90% es clase 1, un modelo que siempre predice 1 ya tiene ~90% accuracy.

En **trading:** Por eso en long-only importan precision y falsos positivos.


> **Actividad B1 (5 min):** si la clase 1 tiene > 55 %, ¿qué accuracy obtiene un modelo que *siempre* predice 1? (Respuesta: ~proporción de unos.) Por eso miramos precision/recall en trading.



<a id='3.4'></a>
## 4.4 Ingeniería de características — indicadores técnicos

### Paso a paso — para qué sirve cada familia (Bitcoin)

| Familia | Variables (ej.) | Lectura en trading |
|---------|-----------------|---------------------|
| **Tendencia** | EMA10/30/200, MA21/63/252 | Precio vs media: ¿vamos arriba o abajo del consenso? |
| **Momentum** | MOM10/30, ROC10/30 | Velocidad del cambio de precio (% o diferencia) |
| **Sobrecompra/sobreventa** | RSI10/30/200 | RSI alto → muchas subidas recientes; bajo → muchas caídas |
| **Estocástico** | %K, %D | Posición del cierre dentro del rango High–Low reciente |
| **Liquidez / nivel** | `Close`, `Volume_(BTC)`, `Weighted_Price` | Contexto de precio y actividad |

A continuación: **un cálculo por celda**. Lee **Objetivo / En palabras simples** antes de ejecutar.


### Paso B4.0a — Definir función EMA

**Objetivo:** Completar «Definir función EMA» y verificar la salida antes de continuar.

**En palabras simples:** Crea la función auxiliar `EMA(df, n)`. En trading: EMA pondera más los cierres recientes que una SMA.

**Qué hace el código:** Crea la función auxiliar `EMA(df, n)`.

**Qué debes ver al ejecutar:** Solo define la función; aún no modifica `dataset`.

**Qué implica para Bitcoin / trading:** EMA pondera más los cierres recientes que una SMA.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Solo define la función; aún no modifica `dataset`.


In [ ]:
def EMA(df, n):
    EMA = pd.Series(df['Close'].ewm(span=n, min_periods=n).mean(), name='EMA_' + str(n))
    return EMA


### Paso B4.1.10 — Indicador `EMA10`

**Objetivo:** Dejar creada la columna `EMA10` en `dataset`.

**En palabras simples:** Convertimos el historial de precios en el número `EMA10`; ese número resume una idea de mercado (tendencia, momentum, etc.).

**Qué hace el código:** `dataset['EMA10'] = EMA(dataset, 10)` usando la función del paso anterior.

**Qué debes ver al ejecutar:** Se imprime el último valor de `EMA10` (número finito, no NaN).

**Qué implica para Bitcoin / trading:** Si Close > EMA10, el precio va por encima de la tendencia muy reciente.

**Qué implica para el modelo ML:** La columna `EMA10` entra en **X**; el Random Forest la usará junto con las demás para predecir si `signal` es 0 o 1.

**Confirma por escrito (cuaderno o chat):** Último `EMA10` = _____ (rellena). ¿Tiene sentido frente al Close actual?


In [ ]:
dataset['EMA10'] = EMA(dataset, 10)
print('EMA10 (último):', round(float(dataset['EMA10'].iloc[-1]), 2))


### Paso B4.1.30 — Indicador `EMA30`

**Objetivo:** Dejar creada la columna `EMA30` en `dataset`.

**En palabras simples:** Convertimos el historial de precios en el número `EMA30`; ese número resume una idea de mercado (tendencia, momentum, etc.).

**Qué hace el código:** `dataset['EMA30'] = EMA(dataset, 30)` usando la función del paso anterior.

**Qué debes ver al ejecutar:** Se imprime el último valor de `EMA30` (número finito, no NaN).

**Qué implica para Bitcoin / trading:** Compara el precio con el consenso de ~30 minutos.

**Qué implica para el modelo ML:** La columna `EMA30` entra en **X**; el Random Forest la usará junto con las demás para predecir si `signal` es 0 o 1.

**Confirma por escrito (cuaderno o chat):** Último `EMA30` = _____ (rellena). ¿Tiene sentido frente al Close actual?


In [ ]:
dataset['EMA30'] = EMA(dataset, 30)
print('EMA30 (último):', round(float(dataset['EMA30'].iloc[-1]), 2))


### Paso B4.1.200 — Indicador `EMA200`

**Objetivo:** Dejar creada la columna `EMA200` en `dataset`.

**En palabras simples:** Convertimos el historial de precios en el número `EMA200`; ese número resume una idea de mercado (tendencia, momentum, etc.).

**Qué hace el código:** `dataset['EMA200'] = EMA(dataset, 200)` usando la función del paso anterior.

**Qué debes ver al ejecutar:** Se imprime el último valor de `EMA200` (número finito, no NaN).

**Qué implica para Bitcoin / trading:** Tendencia lenta: útil para ver si el minuto actual va ‘con’ o ‘contra’ el tramo largo.

**Qué implica para el modelo ML:** La columna `EMA200` entra en **X**; el Random Forest la usará junto con las demás para predecir si `signal` es 0 o 1.

**Confirma por escrito (cuaderno o chat):** Último `EMA200` = _____ (rellena). ¿Tiene sentido frente al Close actual?


In [ ]:
dataset['EMA200'] = EMA(dataset, 200)
print('EMA200 (último):', round(float(dataset['EMA200'].iloc[-1]), 2))


### Paso B4.2a — Definir función ROC

**Objetivo:** Completar «Definir función ROC» y verificar la salida antes de continuar.

**En palabras simples:** Rate of change en porcentaje respecto a hace n minutos. En trading: ROC > 0 → precio subió vs hace n minutos.

**Qué hace el código:** Rate of change en porcentaje respecto a hace n minutos.

**Qué debes ver al ejecutar:** Función `ROC` definida.

**Qué implica para Bitcoin / trading:** ROC > 0 → precio subió vs hace n minutos.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Función `ROC` definida.


In [ ]:
def ROC(df, n):
    M = df.diff(n - 1)
    N = df.shift(n - 1)
    ROC = pd.Series(((M / N) * 100), name='ROC_' + str(n))
    return ROC


### Paso B4.2.10 — Indicador `ROC10`

**Objetivo:** Dejar creada la columna `ROC10` en `dataset`.

**En palabras simples:** Convertimos el historial de precios en el número `ROC10`; ese número resume una idea de mercado (tendencia, momentum, etc.).

**Qué hace el código:** Porcentaje de cambio del Close respecto a hace 10 minutos.

**Qué debes ver al ejecutar:** Se imprime el último valor de `ROC10` (número finito, no NaN).

**Qué implica para Bitcoin / trading:** ROC10 > 0 ⇒ el precio subió en ese horizonte; ROC10 < 0 ⇒ bajó.

**Qué implica para el modelo ML:** La columna `ROC10` entra en **X**; el Random Forest la usará junto con las demás para predecir si `signal` es 0 o 1.

**Confirma por escrito (cuaderno o chat):** Último `ROC10` = _____ (rellena). ¿Tiene sentido frente al Close actual?


In [ ]:
dataset['ROC10'] = ROC(dataset['Close'], 10)
print('ROC10 (último):', round(float(dataset['ROC10'].iloc[-1]), 3))


### Paso B4.2.30 — Indicador `ROC30`

**Objetivo:** Dejar creada la columna `ROC30` en `dataset`.

**En palabras simples:** Convertimos el historial de precios en el número `ROC30`; ese número resume una idea de mercado (tendencia, momentum, etc.).

**Qué hace el código:** Porcentaje de cambio del Close respecto a hace 30 minutos.

**Qué debes ver al ejecutar:** Se imprime el último valor de `ROC30` (número finito, no NaN).

**Qué implica para Bitcoin / trading:** ROC30 > 0 ⇒ el precio subió en ese horizonte; ROC30 < 0 ⇒ bajó.

**Qué implica para el modelo ML:** La columna `ROC30` entra en **X**; el Random Forest la usará junto con las demás para predecir si `signal` es 0 o 1.

**Confirma por escrito (cuaderno o chat):** Último `ROC30` = _____ (rellena). ¿Tiene sentido frente al Close actual?


In [ ]:
dataset['ROC30'] = ROC(dataset['Close'], 30)
print('ROC30 (último):', round(float(dataset['ROC30'].iloc[-1]), 3))


### Paso B4.3a — Definir función MOM

**Objetivo:** Completar «Definir función MOM» y verificar la salida antes de continuar.

**En palabras simples:** Momentum = diferencia de precio en n minutos. En trading: Similar a ROC pero en unidades de precio, no %.

**Qué hace el código:** Momentum = diferencia de precio en n minutos.

**Qué debes ver al ejecutar:** Función `MOM` definida.

**Qué implica para Bitcoin / trading:** Similar a ROC pero en unidades de precio, no %.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Función `MOM` definida.


In [ ]:
def MOM(df, n):
    MOM = pd.Series(df.diff(n), name='Momentum_' + str(n))
    return MOM


### Paso B4.3.10 — Indicador `MOM10`

**Objetivo:** Dejar creada la columna `MOM10` en `dataset`.

**En palabras simples:** Convertimos el historial de precios en el número `MOM10`; ese número resume una idea de mercado (tendencia, momentum, etc.).

**Qué hace el código:** Diferencia de precio Close(t) − Close(t−10).

**Qué debes ver al ejecutar:** Se imprime el último valor de `MOM10` (número finito, no NaN).

**Qué implica para Bitcoin / trading:** MOM10 positivo ⇒ precio subió 10 minutos; negativo ⇒ bajó.

**Qué implica para el modelo ML:** La columna `MOM10` entra en **X**; el Random Forest la usará junto con las demás para predecir si `signal` es 0 o 1.

**Confirma por escrito (cuaderno o chat):** Último `MOM10` = _____ (rellena). ¿Tiene sentido frente al Close actual?


In [ ]:
dataset['MOM10'] = MOM(dataset['Close'], 10)
print('MOM10 (último):', round(float(dataset['MOM10'].iloc[-1]), 2))


### Paso B4.3.30 — Indicador `MOM30`

**Objetivo:** Dejar creada la columna `MOM30` en `dataset`.

**En palabras simples:** Convertimos el historial de precios en el número `MOM30`; ese número resume una idea de mercado (tendencia, momentum, etc.).

**Qué hace el código:** Diferencia de precio Close(t) − Close(t−30).

**Qué debes ver al ejecutar:** Se imprime el último valor de `MOM30` (número finito, no NaN).

**Qué implica para Bitcoin / trading:** MOM30 positivo ⇒ precio subió 30 minutos; negativo ⇒ bajó.

**Qué implica para el modelo ML:** La columna `MOM30` entra en **X**; el Random Forest la usará junto con las demás para predecir si `signal` es 0 o 1.

**Confirma por escrito (cuaderno o chat):** Último `MOM30` = _____ (rellena). ¿Tiene sentido frente al Close actual?


In [ ]:
dataset['MOM30'] = MOM(dataset['Close'], 30)
print('MOM30 (último):', round(float(dataset['MOM30'].iloc[-1]), 2))


### Paso B4.4a — Definir función RSI

**Objetivo:** Completar «Definir función RSI» y verificar la salida antes de continuar.

**En palabras simples:** Implementación RSI del repo (0–100). En trading: RSI alto → dominan subidas recientes; bajo → dominan caídas.

**Qué hace el código:** Implementación RSI del repo (0–100).

**Qué debes ver al ejecutar:** Función `RSI` definida.

**Qué implica para Bitcoin / trading:** RSI alto → dominan subidas recientes; bajo → dominan caídas.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Función `RSI` definida.


In [ ]:
def RSI(series, period):
    delta = series.diff().dropna()
    u = delta * 0
    d = u.copy()
    u[delta > 0] = delta[delta > 0]
    d[delta < 0] = -delta[delta < 0]
    u[u.index[period - 1]] = np.mean(u[:period])
    u = u.drop(u.index[:(period - 1)])
    d[d.index[period - 1]] = np.mean(d[:period])
    d = d.drop(d.index[:(period - 1)])
    rs = u.ewm(com=period - 1, adjust=False).mean() / d.ewm(com=period - 1, adjust=False).mean()
    return 100 - 100 / (1 + rs)


### Paso B4.4.10 — Indicador `RSI10`

**Objetivo:** Dejar creada la columna `RSI10` en `dataset`.

**En palabras simples:** Convertimos el historial de precios en el número `RSI10`; ese número resume una idea de mercado (tendencia, momentum, etc.).

**Qué hace el código:** Índice de fuerza relativa con ventana 10 (fórmula del repo).

**Qué debes ver al ejecutar:** Se imprime el último valor de `RSI10` (número finito, no NaN).

**Qué implica para Bitcoin / trading:** RSI10 cerca de 70+ ⇒ muchas subidas recientes; cerca de 30− ⇒ muchas caídas (lectura clásica).

**Qué implica para el modelo ML:** La columna `RSI10` entra en **X**; el Random Forest la usará junto con las demás para predecir si `signal` es 0 o 1.

**Confirma por escrito (cuaderno o chat):** Último `RSI10` = _____ (rellena). ¿Tiene sentido frente al Close actual?


In [ ]:
dataset['RSI10'] = RSI(dataset['Close'], 10)
print('RSI10 (último):', round(float(dataset['RSI10'].iloc[-1]), 2))


### Paso B4.4.30 — Indicador `RSI30`

**Objetivo:** Dejar creada la columna `RSI30` en `dataset`.

**En palabras simples:** Convertimos el historial de precios en el número `RSI30`; ese número resume una idea de mercado (tendencia, momentum, etc.).

**Qué hace el código:** Índice de fuerza relativa con ventana 30 (fórmula del repo).

**Qué debes ver al ejecutar:** Se imprime el último valor de `RSI30` (número finito, no NaN).

**Qué implica para Bitcoin / trading:** RSI30 cerca de 70+ ⇒ muchas subidas recientes; cerca de 30− ⇒ muchas caídas (lectura clásica).

**Qué implica para el modelo ML:** La columna `RSI30` entra en **X**; el Random Forest la usará junto con las demás para predecir si `signal` es 0 o 1.

**Confirma por escrito (cuaderno o chat):** Último `RSI30` = _____ (rellena). ¿Tiene sentido frente al Close actual?


In [ ]:
dataset['RSI30'] = RSI(dataset['Close'], 30)
print('RSI30 (último):', round(float(dataset['RSI30'].iloc[-1]), 2))


### Paso B4.4.200 — Indicador `RSI200`

**Objetivo:** Dejar creada la columna `RSI200` en `dataset`.

**En palabras simples:** Convertimos el historial de precios en el número `RSI200`; ese número resume una idea de mercado (tendencia, momentum, etc.).

**Qué hace el código:** Índice de fuerza relativa con ventana 200 (fórmula del repo).

**Qué debes ver al ejecutar:** Se imprime el último valor de `RSI200` (número finito, no NaN).

**Qué implica para Bitcoin / trading:** RSI200 cerca de 70+ ⇒ muchas subidas recientes; cerca de 30− ⇒ muchas caídas (lectura clásica).

**Qué implica para el modelo ML:** La columna `RSI200` entra en **X**; el Random Forest la usará junto con las demás para predecir si `signal` es 0 o 1.

**Confirma por escrito (cuaderno o chat):** Último `RSI200` = _____ (rellena). ¿Tiene sentido frente al Close actual?


In [ ]:
dataset['RSI200'] = RSI(dataset['Close'], 200)
print('RSI200 (último):', round(float(dataset['RSI200'].iloc[-1]), 2))


### Paso B4.5a — Funciones estocástico %K y %D

**Objetivo:** Completar «Funciones estocástico %K y %D» y verificar la salida antes de continuar.

**En palabras simples:** Posición del cierre en el rango High–Low reciente. En trading: %K reacciona rápido; %D es media móvil de %K (más suave).

**Qué hace el código:** Posición del cierre en el rango High–Low reciente.

**Qué debes ver al ejecutar:** Funciones `STOK` y `STOD` definidas.

**Qué implica para Bitcoin / trading:** %K reacciona rápido; %D es media móvil de %K (más suave).

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Funciones `STOK` y `STOD` definidas.


In [ ]:
def STOK(close, low, high, n):
    STOK = ((close - low.rolling(n).min()) / (high.rolling(n).max() - low.rolling(n).min())) * 100
    return STOK

def STOD(close, low, high, n):
    STOK = ((close - low.rolling(n).min()) / (high.rolling(n).max() - low.rolling(n).min())) * 100
    STOD = STOK.rolling(3).mean()
    return STOD


### Paso B4.5.10K — %K10

**Objetivo:** Completar «%K10» y verificar la salida antes de continuar.

**En palabras simples:** Estocástico %K con ventana 10.

**Qué hace el código:** Estocástico %K con ventana 10.

**Qué debes ver al ejecutar:** Último %K10 (0–100).

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Último %K10 (0–100).


In [ ]:
dataset['%K10'] = STOK(dataset['Close'], dataset['Low'], dataset['High'], 10)
print('%K10 (último):', round(float(dataset['%K10'].iloc[-1]), 2))


### Paso B4.5.10D — %D10

**Objetivo:** Completar «%D10» y verificar la salida antes de continuar.

**En palabras simples:** Suavizado %D (media de %K, ventana 3).

**Qué hace el código:** Suavizado %D (media de %K, ventana 3).

**Qué debes ver al ejecutar:** Último %D10.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Último %D10.


In [ ]:
dataset['%D10'] = STOD(dataset['Close'], dataset['Low'], dataset['High'], 10)
print('%D10 (último):', round(float(dataset['%D10'].iloc[-1]), 2))


### Paso B4.5.30K — %K30

**Objetivo:** Completar «%K30» y verificar la salida antes de continuar.

**En palabras simples:** Estocástico %K con ventana 30.

**Qué hace el código:** Estocástico %K con ventana 30.

**Qué debes ver al ejecutar:** Último %K30 (0–100).

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Último %K30 (0–100).


In [ ]:
dataset['%K30'] = STOK(dataset['Close'], dataset['Low'], dataset['High'], 30)
print('%K30 (último):', round(float(dataset['%K30'].iloc[-1]), 2))


### Paso B4.5.30D — %D30

**Objetivo:** Completar «%D30» y verificar la salida antes de continuar.

**En palabras simples:** Suavizado %D (media de %K, ventana 3).

**Qué hace el código:** Suavizado %D (media de %K, ventana 3).

**Qué debes ver al ejecutar:** Último %D30.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Último %D30.


In [ ]:
dataset['%D30'] = STOD(dataset['Close'], dataset['Low'], dataset['High'], 30)
print('%D30 (último):', round(float(dataset['%D30'].iloc[-1]), 2))


### Paso B4.5.200K — %K200

**Objetivo:** Completar «%K200» y verificar la salida antes de continuar.

**En palabras simples:** Estocástico %K con ventana 200.

**Qué hace el código:** Estocástico %K con ventana 200.

**Qué debes ver al ejecutar:** Último %K200 (0–100).

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Último %K200 (0–100).


In [ ]:
dataset['%K200'] = STOK(dataset['Close'], dataset['Low'], dataset['High'], 200)
print('%K200 (último):', round(float(dataset['%K200'].iloc[-1]), 2))


### Paso B4.5.200D — %D200

**Objetivo:** Completar «%D200» y verificar la salida antes de continuar.

**En palabras simples:** Suavizado %D (media de %K, ventana 3).

**Qué hace el código:** Suavizado %D (media de %K, ventana 3).

**Qué debes ver al ejecutar:** Último %D200.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Último %D200.


In [ ]:
dataset['%D200'] = STOD(dataset['Close'], dataset['Low'], dataset['High'], 200)
print('%D200 (último):', round(float(dataset['%D200'].iloc[-1]), 2))


### Paso B4.6a — Definir función MA

**Objetivo:** Completar «Definir función MA» y verificar la salida antes de continuar.

**En palabras simples:** Media móvil simple con `min_periods=n`.

**Qué hace el código:** Media móvil simple con `min_periods=n`.

**Qué debes ver al ejecutar:** Función `MA` definida.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Función `MA` definida.


In [ ]:
def MA(df, n):
    MA = pd.Series(df['Close'].rolling(n, min_periods=n).mean(), name='MA_' + str(n))
    return MA


### Paso B4.6.MA21 — Columna MA21

**Objetivo:** Completar «Columna MA21» y verificar la salida antes de continuar.

**En palabras simples:** MA simple con ventana 10 minutos. En trading: Nombre MA21 en repo; ventana 10 min.

**Qué hace el código:** MA simple con ventana 10 minutos.

**Qué debes ver al ejecutar:** Último valor de MA21.

**Qué implica para Bitcoin / trading:** Nombre MA21 en repo; ventana 10 min.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Último valor de MA21.


In [ ]:
dataset['MA21'] = MA(dataset, 10)
print('MA21 (último):', round(float(dataset['MA21'].iloc[-1]), 2))


### Paso B4.6.MA63 — Columna MA63

**Objetivo:** Completar «Columna MA63» y verificar la salida antes de continuar.

**En palabras simples:** MA simple con ventana 30 minutos. En trading: Nombre MA63; ventana 30 min.

**Qué hace el código:** MA simple con ventana 30 minutos.

**Qué debes ver al ejecutar:** Último valor de MA63.

**Qué implica para Bitcoin / trading:** Nombre MA63; ventana 30 min.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Último valor de MA63.


In [ ]:
dataset['MA63'] = MA(dataset, 30)
print('MA63 (último):', round(float(dataset['MA63'].iloc[-1]), 2))


### Paso B4.6.MA252 — Columna MA252

**Objetivo:** Completar «Columna MA252» y verificar la salida antes de continuar.

**En palabras simples:** MA simple con ventana 200 minutos. En trading: Nombre MA252; ventana 200 min.

**Qué hace el código:** MA simple con ventana 200 minutos.

**Qué debes ver al ejecutar:** Último valor de MA252.

**Qué implica para Bitcoin / trading:** Nombre MA252; ventana 200 min.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Último valor de MA252.


In [ ]:
dataset['MA252'] = MA(dataset, 200)
print('MA252 (último):', round(float(dataset['MA252'].iloc[-1]), 2))


### Paso B4.7 — Revisión del dataset enriquecido

**Objetivo:** Completar «Revisión del dataset enriquecido» y verificar la salida antes de continuar.

**En palabras simples:** Muestra las últimas filas con precio, señal e indicadores.

**Qué hace el código:** Muestra las últimas filas con precio, señal e indicadores.

**Qué debes ver al ejecutar:** Muchas columnas; verifica que no haya NaN al final.

**Qué implica para el modelo ML:** Tras `dropna` perderemos filas iniciales con ventanas incompletas.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Muchas columnas; verifica que no haya NaN al final.


In [ ]:
print('Número de columnas:', len(dataset.columns))
dataset.tail(3)


<a id='3.5'></a>
## 4.5 Visualización de datos

Exploramos relaciones lineales entre indicadores (como en el caso Bitcoin del repo).


### Paso B5.1 — Eliminar columnas no predictivas

**Objetivo:** Completar «Eliminar columnas no predictivas» y verificar la salida antes de continuar.

**En palabras simples:** Quita OHLC crudo, volumen fiat y SMA de etiquetado. En trading: Mantenemos `Close`, volumen BTC y features derivados.

**Qué hace el código:** Quita OHLC crudo, volumen fiat y SMA de etiquetado.

**Qué debes ver al ejecutar:** Lista de columnas restantes (incluye `signal`).

**Qué implica para Bitcoin / trading:** Mantenemos `Close`, volumen BTC y features derivados.

**Qué implica para el modelo ML:** Evita que el modelo copie trivialmente la regla SMA vía `short_mavg`.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Lista de columnas restantes (incluye `signal`).


In [ ]:
dataset = dataset.drop(['High', 'Low', 'Open', 'Volume_(Currency)', 'short_mavg', 'long_mavg'], axis=1)
print('Columnas tras drop:', list(dataset.columns))


### Paso B5.2 — Eliminar filas con NaN

**Objetivo:** Completar «Eliminar filas con NaN» y verificar la salida antes de continuar.

**En palabras simples:** Borra filas donde algún indicador aún no está definido.

**Qué hace el código:** Borra filas donde algún indicador aún no está definido.

**Qué debes ver al ejecutar:** Cuántas filas se eliminaron vs antes.

**Qué implica para el modelo ML:** Primeras filas tras ventanas largas (p. ej. 200 min) suelen ser NaN.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Cuántas filas se eliminaron vs antes.


In [ ]:
filas_antes = len(dataset)
dataset = dataset.dropna(axis=0)
print('Filas eliminadas por NaN:', filas_antes - len(dataset))
print('Filas listas para modelado:', len(dataset))


### Paso B5.3a — Matriz de correlación (números)

**Objetivo:** Completar «Matriz de correlación (números)» y verificar la salida antes de continuar.

**En palabras simples:** Calcula correlación lineal entre todas las columnas numéricas.

**Qué hace el código:** Calcula correlación lineal entre todas las columnas numéricas.

**Qué debes ver al ejecutar:** Objeto `correlation` (usa la celda siguiente para graficar).

**Qué implica para el modelo ML:** Correlaciones |r|>0.9 sugieren redundancia entre features.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Objeto `correlation` (usa la celda siguiente para graficar).


In [ ]:
correlation = dataset.corr()
print('Tamaño matriz:', correlation.shape)


### Paso B5.3b — Heatmap de correlación

**Objetivo:** Completar «Heatmap de correlación» y verificar la salida antes de continuar.

**En palabras simples:** Visualiza la matriz (como el caso Bitcoin del repo). En trading: EMA/MA/EMA200 a menudo correlacionan fuerte con Close.

**Qué hace el código:** Visualiza la matriz (como el caso Bitcoin del repo).

**Qué debes ver al ejecutar:** Bloques rojos/azules = variables que se mueven juntas.

**Qué implica para Bitcoin / trading:** EMA/MA/EMA200 a menudo correlacionan fuerte con Close.

**Qué implica para el modelo ML:** LDA/LR sufren con colinealidad; RF/GBM la toleran mejor.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Bloques rojos/azules = variables que se mueven juntas.


In [ ]:
figsize = (10, 10) if MODO_LIGERO else (15, 15)
plt.figure(figsize=figsize)
plt.title('Matriz de correlación')
sns.heatmap(correlation, vmax=1, square=True, annot=False, cmap='cubehelix')
plt.show()


<a id='3.6'></a>
## 4.6 Selección de variables

Aquí **mantenemos** precio, volumen e indicadores calculados, y excluimos columnas crudas ya reemplazadas por features. La variable objetivo es `signal`.


<a id='3.7'></a>
## 4.7 Transformación — estandarización

**StandardScaler** (media 0, varianza 1) ayuda a modelos sensibles a escala (KNN, redes). Aquí se usa en el **Grid Search** del Random Forest; el **modelo final** se entrena sobre `X_train` **sin escalar**.


**✓ Fin de la parte B — checklist explícito**

- [ ] Existe columna **`signal`** (0/1) y sé qué significa en trading.
- [ ] Calculé indicadores; **`short_mavg` / `long_mavg` ya NO están** en X (las eliminamos).
- [ ] Sé cuántas filas quedan tras `dropna`.

Si algún ítem no lo tienes claro, **vuelve al paso B3 o B5** antes de la parte C.


---
## Parte C

**Qué hacemos aquí (sin jerga):** Separamos **Y** y **X**, partimos train/validation y **comparamos algoritmos** con validación cruzada (sin mirar validation todavía para elegir).

**Al terminar la parte C, debes poder decir en voz alta:** Qué es train vs validation, qué mide CV, y cuál modelo candidato llevar al Grid Search (casi siempre RF).

**Pregunta guía:** ¿Por qué no entrenamos y evaluamos en las mismas filas si queremos saber si funciona en el futuro?


<a id='4'></a>
# 5. Evaluar algoritmos y modelos

### Vista unificada: del precio Bitcoin a la predicción

```
Precio minuto a minuto → indicadores X_t → modelo → ŷ_t (0/1)
                                              ↓
                         comparar con signal_t (SMA) en validación
                                              ↓
                         backtest: retorno × ŷ_{t-1}
```

<a id='4.1'></a>
## 5.1 Partición entrenamiento / validación

### Paso a paso (ML)

1. Tomar las **últimas `N_FILAS`** (mercado reciente).
2. Separar **Y** = `signal`, **X** = resto de columnas numéricas.
3. **80 % train** → ajustar modelos; **20 % validation** → simular “futuro” no visto en el ajuste.
4. `stratify=Y` mantiene proporción de 0/1 en ambos conjuntos.

Usamos `random_state=1` para reproducibilidad.


### Paso C1.1 — Recortar ventana temporal

**Objetivo:** Completar «Recortar ventana temporal» y verificar la salida antes de continuar.

**En palabras simples:** Toma las últimas `N_FILAS` filas del dataset limpio. En trading: Enfoca el modelo en el tramo más reciente de la muestra.

**Qué hace el código:** Toma las últimas `N_FILAS` filas del dataset limpio.

**Qué debes ver al ejecutar:** Valor de `n_use` impreso.

**Qué implica para Bitcoin / trading:** Enfoca el modelo en el tramo más reciente de la muestra.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Valor de `n_use` impreso.


In [ ]:
n_use = min(N_FILAS, len(dataset))
subset_dataset = dataset.iloc[-n_use:]
print('Filas para modelado:', n_use)


### Paso C1.2 — Variable objetivo Y

**Objetivo:** Separar claramente etiqueta vs predictores.

**En palabras simples:** Y es la respuesta correcta del examen; X son las pistas.

**Qué hace el código:** Asigna `Y = subset_dataset['signal']` — solo la columna etiqueta.

**Qué debes ver al ejecutar:** Imprime shape de Y y **P(Y=1)** (proporción de minutos con signal=1).

**Qué implica para Bitcoin / trading:** P(Y=1) alto ⇒ en esta ventana la regla SMA estuvo ‘larga’ la mayor parte del tiempo.

**Qué implica para el modelo ML:** **Y es lo que el modelo debe predecir.** No incluyas Y dentro de X.

**Confirma por escrito (cuaderno o chat):** P(Y=1) en mi muestra = _____ (escribe el número impreso).


In [ ]:
Y = subset_dataset['signal']
print('Y shape:', Y.shape, '| P(Y=1):', round(float(Y.mean()), 3))


### Paso C1.3 — Matriz de features X

**Objetivo:** Completar «Matriz de features X» y verificar la salida antes de continuar.

**En palabras simples:** Todas las columnas excepto `signal`.

**Qué hace el código:** Todas las columnas excepto `signal`.

**Qué debes ver al ejecutar:** Número de features y filas en `X`.

**Qué implica para el modelo ML:** Cada fila es un minuto; cada columna un indicador o precio/volumen.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Número de features y filas en `X`.


In [ ]:
X = subset_dataset.loc[:, subset_dataset.columns != 'signal']
print('X shape:', X.shape)
print('Features:', list(X.columns))


### Paso C1.4 — Partición train / validation

**Objetivo:** Tener cuatro objetos: X_train, X_validation, Y_train, Y_validation.

**En palabras simples:** Separamos el examen en parte para practicar (train) y parte sorpresa (validation).

**Qué hace el código:** `train_test_split(X, Y, test_size=0.2, random_state=1)` — **80% train, 20% validation**.

**Qué debes ver al ejecutar:** Imprime shapes; train debe tener ~80% de las filas de X.

**Qué implica para Bitcoin / trading:** Train = minutos donde ‘estudiamos’; validation = minutos donde ‘probamos’ sin haber estudiado esas filas al entrenar.

**Qué implica para el modelo ML:** **Nunca ajustes el modelo final mirando solo train**; validation es el primer hold-out honesto (aunque el split no es temporal estricto).

**Confirma por escrito (cuaderno o chat):** Train tiene _____ filas; validation tiene _____ filas (escribe los números).


In [ ]:
validation_size = 0.2
X_train, X_validation, Y_train, Y_validation = train_test_split(
    X, Y, test_size=validation_size, random_state=1
)
print('Train:', X_train.shape, 'Validation:', X_validation.shape)


**✓ Celda ejecutada — interpreta así:**

Comprueba que la proporción de Y=1 sea similar en train y validation.

En **ML:** No es split temporal estricto; es un hold-out aleatorio 80/20.


<a id='4.2'></a>
## 5.2 Opciones de prueba y métricas de evaluación


### Paso C2.1 — Número de folds

**Objetivo:** Completar «Número de folds» y verificar la salida antes de continuar.

**En palabras simples:** Copia `N_FOLDS` de la configuración de sesión.

**Qué hace el código:** Copia `N_FOLDS` de la configuración de sesión.

**Qué debes ver al ejecutar:** `num_folds` (3 en clase, 10 en modo completo).

**Confirma por escrito (cuaderno o chat):** La salida coincide con: `num_folds` (3 en clase, 10 en modo completo).


In [ ]:
num_folds = N_FOLDS
print('Folds CV:', num_folds)


### Paso C2.2 — Semilla de CV

**Objetivo:** Completar «Semilla de CV» y verificar la salida antes de continuar.

**En palabras simples:** Fija aleatoriedad del KFold para resultados reproducibles.

**Qué hace el código:** Fija aleatoriedad del KFold para resultados reproducibles.

**Qué debes ver al ejecutar:** `seed` = CV_SEED.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: `seed` = CV_SEED.


In [ ]:
seed = CV_SEED
print('Seed CV:', seed)


### Paso C2.3 — Métrica de scoring

**Objetivo:** Completar «Métrica de scoring» y verificar la salida antes de continuar.

**En palabras simples:** Define qué optimiza CV y Grid Search. En trading: Prueba `'precision'` o `'recall'` en Actividad C1 (long-only).

**Qué hace el código:** Define qué optimiza CV y Grid Search.

**Qué debes ver al ejecutar:** Variable `scoring` (por defecto `accuracy`).

**Qué implica para Bitcoin / trading:** Prueba `'precision'` o `'recall'` en Actividad C1 (long-only).

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Variable `scoring` (por defecto `accuracy`).


In [ ]:
scoring = 'accuracy'
# En clase: descomenta UNA línea y comenta accuracy
# scoring = 'precision'
# scoring = 'recall'
print('Scoring activo:', scoring)


**Métricas de clasificación:**

- **Accuracy:** $\frac{TP+TN}{N}$ — útil si clases balanceadas.
- **Precision (clase 1):** $\frac{TP}{TP+FP}$ — penaliza falsas compras.
- **Recall (clase 1):** $\frac{TP}{TP+FN}$ — penaliza perder subidas.
- **AUC-ROC:** calidad del ranking de probabilidades.

Para estrategias **long-only**, a menudo se discute precision vs recall (ver conclusión del notebook original).

### Paso a paso — validación cruzada (qué hace el código después)

1. Partir **train** en `N_FOLDS` bloques temporales aleatorios (KFold).
2. En cada fold: entrenar en k−1 bloques, medir accuracy en el bloque restante.
3. Promediar → estimación de error **sin usar validation**.
4. Repetir para cada algoritmo → **boxplot** al final de la parte C.


<a id='4.3'></a>
## 5.3 Guía de modelos e hiperparámetros

### Paso a paso — algoritmos en **modo clase** (qué aprende cada uno)

| Modelo | Entrada | Salida | En Bitcoin, intuitivamente… |
|--------|---------|--------|-----------------------------|
| **LR** | Vector $X_t$ | $P(\text{compra})$ | Combinación lineal de RSI, ROC… → probabilidad de régimen alcista |
| **LDA** | $X_t$ | Clase 0/1 | Frontera lineal; baseline rápido |
| **CART** | $X_t$ | 0/1 | Reglas del tipo “si RSI < 30 y ROC > 0 → compra” |
| **GBM** | $X_t$ | 0/1 | Corrige errores de árboles pequeños secuencialmente |
| **RF** | $X_t$ | 0/1 | Muchos árboles en submuestras → robustez al ruido minuto a minuto |

Con **`MODO_LIGERO = False`** se añaden KNN, NB, NN y AdaBoost al benchmark.

| Código | Modelo | Idea | Hiperparámetro clave | Si lo subes demasiado… |
|--------|--------|-------------------|----------------------|-------------------------|
| LR | Regresión logística | $P(y=1|x)=\sigma(w^\top x+b)$ | `C` (regularización) | Sobreajuste, coeficientes explosivos |
| LDA | Discriminante lineal | Frontera lineal, covarianza compartida | — | Modelo rígido |
| KNN | k-vecinos | Voto local | `n_neighbors` | Frontera ruidosa si k es pequeño |
| CART | Árbol | Particiones por impureza (Gini/entropía) | `max_depth` | Memoriza ruido minuto a minuto |
| NB | Naive Bayes | Independencia condicional | — | Sesgo si features correlacionadas |
| NN | MLP | Capas ocultas no lineales | `hidden_layer_sizes`, `alpha` | Más capacidad → más riesgo de sobreajuste |
| AB / GBM | Boosting | Suma de clasificadores débiles | `n_estimators`, `learning_rate` | Mejora CV hasta plateau |
| RF | Random Forest | Bagging de árboles | `n_estimators`, `max_depth`, `criterion` | Más árboles → menor varianza |

**Demostración visual** (dos features para poder graficar): efecto de `max_depth` (árbol) y `C` (logística).


> **Pausa docente (2 min):** enseñar un gráfico train vs CV antes de seguir al benchmark.


### Paso C3.0 — Submuestra 2D para gráficos

**Objetivo:** Completar «Submuestra 2D para gráficos» y verificar la salida antes de continuar.

**En palabras simples:** Solo RSI10 y ROC10 + signal (visualización de hiperparámetros).

**Qué hace el código:** Solo RSI10 y ROC10 + signal (visualización de hiperparámetros).

**Qué debes ver al ejecutar:** Tamaño de `Xd_tr`.

**Qué implica para el modelo ML:** Reducir a 2 features permite interpretar curvas de validación.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Tamaño de `Xd_tr`.


In [ ]:
from sklearn.pipeline import Pipeline

demo_cols = ['RSI10', 'ROC10']
demo = subset_dataset[demo_cols + ['signal']].dropna()
Xd = demo[demo_cols]
yd = demo['signal']
Xd_tr, _, yd_tr, _ = train_test_split(Xd, yd, test_size=0.3, random_state=1, stratify=yd)
print('Demo 2D — filas train:', len(Xd_tr))


### Paso C3.1 — Curva de validación — árbol (max_depth)

**Objetivo:** Completar «Curva de validación — árbol (max_depth)» y verificar la salida antes de continuar.

**En palabras simples:** Entrena árboles con distintas profundidades; compara train vs CV. En trading: Profundidad alta = reglas muy específicas al ruido del minuto.

**Qué hace el código:** Entrena árboles con distintas profundidades; compara train vs CV.

**Qué debes ver al ejecutar:** Gráfico: si train sube y CV baja → sobreajuste.

**Qué implica para Bitcoin / trading:** Profundidad alta = reglas muy específicas al ruido del minuto.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Gráfico: si train sube y CV baja → sobreajuste.


In [ ]:
depths = [2, 4, 6, 8, 10, 15] if MODO_LIGERO else [2, 3, 4, 5, 6, 8, 10, 15, 20]
pipe = Pipeline([('sc', StandardScaler()), ('clf', DecisionTreeClassifier(random_state=1))])
tr, va = validation_curve(pipe, Xd_tr, yd_tr, param_name='clf__max_depth', param_range=depths, cv=3, scoring='accuracy')
plt.figure(figsize=(8, 4))
plt.plot(depths, tr.mean(1), 'o-', label='Train')
plt.plot(depths, va.mean(1), 'o-', label='CV')
plt.xlabel('max_depth'); plt.ylabel('Accuracy'); plt.title('Árbol: profundidad vs desempeño')
plt.legend(); plt.show()


### Paso C3.2 — Curva de validación — logística (C)

**Objetivo:** Completar «Curva de validación — logística (C)» y verificar la salida antes de continuar.

**En palabras simples:** Varía regularización `C` en escala logarítmica.

**Qué hace el código:** Varía regularización `C` en escala logarítmica.

**Qué debes ver al ejecutar:** Gráfico semilogx: busca C donde CV sea alta y train no dispare.

**Qué implica para el modelo ML:** C grande → modelo más flexible; C pequeño → más regularización.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Gráfico semilogx: busca C donde CV sea alta y train no dispare.


In [ ]:
Cs = np.logspace(-3, 2, 8)
pipe_lr = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=3000, random_state=1))])
tr, va = validation_curve(pipe_lr, Xd_tr, yd_tr, param_name='clf__C', param_range=Cs, cv=3, scoring='accuracy')
plt.figure(figsize=(8, 4))
plt.semilogx(Cs, tr.mean(1), 'o-', label='Train')
plt.semilogx(Cs, va.mean(1), 'o-', label='CV')
plt.xlabel('C'); plt.title('Logística: regularización'); plt.legend(); plt.show()


**Cómo leer los gráficos:** si la curva de *train* sube y la de *CV* cae, estás **sobreajustando** (modelo memoriza). Elige hiperparámetros donde CV esté alta y la brecha train–CV sea moderada.


<a id='4.4'></a>
## 5.4 Comparar modelos

### Paso a paso — lectura del benchmark

1. Ejecutar la celda que define la **lista `models`**.
2. Ejecutar el **bucle CV**: imprime `media (desv. estándar)` por modelo.
3. Ejecutar el **boxplot**: cada caja = distribución de accuracy en folds.
4. **Elegir candidato** para Grid Search (suele ser Random Forest).

- **Modo clase:** 5 modelos (~8–12 min Colab).
- **Modo completo:** 9 modelos.


### Paso C4.1 — Inicializar benchmark CV

**Objetivo:** Completar «Inicializar benchmark CV» y verificar la salida antes de continuar.

**En palabras simples:** Crea listas vacías y un KFold compartido para todos los modelos.

**Qué hace el código:** Crea listas vacías y un KFold compartido para todos los modelos.

**Qué debes ver al ejecutar:** Mensaje con número de folds.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Mensaje con número de folds.


In [ ]:
results = []
names = []
kfold = KFold(n_splits=num_folds, shuffle=True, random_state=seed)
print('Benchmark CV — folds:', num_folds, '| scoring:', scoring)


### Paso C4.2 — Lista de modelos (modo sesión)

**Objetivo:** Completar «Lista de modelos (modo sesión)» y verificar la salida antes de continuar.

**En palabras simples:** Muestra qué algoritmos se evaluarán según `MODO_LIGERO`.

**Qué hace el código:** Muestra qué algoritmos se evaluarán según `MODO_LIGERO`.

**Qué debes ver al ejecutar:** Lista impresa de códigos LR, LDA, …

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Lista impresa de códigos LR, LDA, …


In [ ]:
if MODO_LIGERO:
    bench_list = [
        ('LR', LogisticRegression(max_iter=3000)),
        ('LDA', LinearDiscriminantAnalysis()),
        ('CART', DecisionTreeClassifier(max_depth=12, random_state=1)),
        ('GBM', GradientBoostingClassifier(n_estimators=50, random_state=1)),
        ('RF', RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=1)),
    ]
else:
    bench_list = [
        ('LR', LogisticRegression(max_iter=3000)),
        ('LDA', LinearDiscriminantAnalysis()),
        ('KNN', KNeighborsClassifier()),
        ('CART', DecisionTreeClassifier()),
        ('NB', GaussianNB()),
        ('NN', MLPClassifier(max_iter=400)),
        ('AB', AdaBoostClassifier()),
        ('GBM', GradientBoostingClassifier()),
        ('RF', RandomForestClassifier(n_jobs=-1)),
    ]
print('Modelos:', [b[0] for b in bench_list])


### Paso C4.3-LR — CV — LR

**Objetivo:** Completar «CV — LR» y verificar la salida antes de continuar.

**En palabras simples:** Un fold a la vez: entrena **LR** en k−1 particiones y mide la métrica `scoring` en la restante. En trading: P(compra) ~ w·indicadores.

**Qué hace el código:** Un fold a la vez: entrena **LR** en k−1 particiones y mide la métrica `scoring` en la restante.

**Qué debes ver al ejecutar:** Línea `LR: media (std)` — anota la media.

**Qué implica para Bitcoin / trading:** P(compra) ~ w·indicadores.

**Qué implica para el modelo ML:** Frontera lineal; baseline.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Línea `LR: media (std)` — anota la media.


In [ ]:
_found = False
for _code, _model in bench_list:
    if _code == 'LR':
        cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
        results.append(cv_results)
        names.append('LR')
        print('LR: %f (%f)' % (cv_results.mean(), cv_results.std()))
        _found = True
        break
if not _found:
    print('LR: no incluido en bench_list (omitido).')


### Paso C4.3-LDA — CV — LDA

**Objetivo:** Completar «CV — LDA» y verificar la salida antes de continuar.

**En palabras simples:** Un fold a la vez: entrena **LDA** en k−1 particiones y mide la métrica `scoring` en la restante. En trading: Baseline rápido en alta dimensión.

**Qué hace el código:** Un fold a la vez: entrena **LDA** en k−1 particiones y mide la métrica `scoring` en la restante.

**Qué debes ver al ejecutar:** Línea `LDA: media (std)` — anota la media.

**Qué implica para Bitcoin / trading:** Baseline rápido en alta dimensión.

**Qué implica para el modelo ML:** Frontera lineal LDA.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Línea `LDA: media (std)` — anota la media.


In [ ]:
_found = False
for _code, _model in bench_list:
    if _code == 'LDA':
        cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
        results.append(cv_results)
        names.append('LDA')
        print('LDA: %f (%f)' % (cv_results.mean(), cv_results.std()))
        _found = True
        break
if not _found:
    print('LDA: no incluido en bench_list (omitido).')


### Paso C4.3-CART — CV — CART

**Objetivo:** Completar «CV — CART» y verificar la salida antes de continuar.

**En palabras simples:** Un fold a la vez: entrena **CART** en k−1 particiones y mide la métrica `scoring` en la restante. En trading: Reglas locales minuto a minuto.

**Qué hace el código:** Un fold a la vez: entrena **CART** en k−1 particiones y mide la métrica `scoring` en la restante.

**Qué debes ver al ejecutar:** Línea `CART: media (std)` — anota la media.

**Qué implica para Bitcoin / trading:** Reglas locales minuto a minuto.

**Qué implica para el modelo ML:** Árbol prof. limitada 12.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Línea `CART: media (std)` — anota la media.


In [ ]:
_found = False
for _code, _model in bench_list:
    if _code == 'CART':
        cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
        results.append(cv_results)
        names.append('CART')
        print('CART: %f (%f)' % (cv_results.mean(), cv_results.std()))
        _found = True
        break
if not _found:
    print('CART: no incluido en bench_list (omitido).')


### Paso C4.3-GBM — CV — GBM

**Objetivo:** Completar «CV — GBM» y verificar la salida antes de continuar.

**En palabras simples:** Un fold a la vez: entrena **GBM** en k−1 particiones y mide la métrica `scoring` en la restante. En trading: Corrige errores de modelos previos.

**Qué hace el código:** Un fold a la vez: entrena **GBM** en k−1 particiones y mide la métrica `scoring` en la restante.

**Qué debes ver al ejecutar:** Línea `GBM: media (std)` — anota la media.

**Qué implica para Bitcoin / trading:** Corrige errores de modelos previos.

**Qué implica para el modelo ML:** 50 árboles débiles secuenciales.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Línea `GBM: media (std)` — anota la media.


In [ ]:
_found = False
for _code, _model in bench_list:
    if _code == 'GBM':
        cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
        results.append(cv_results)
        names.append('GBM')
        print('GBM: %f (%f)' % (cv_results.mean(), cv_results.std()))
        _found = True
        break
if not _found:
    print('GBM: no incluido en bench_list (omitido).')


### Paso C4.3-RF — CV — RF

**Objetivo:** Completar «CV — RF» y verificar la salida antes de continuar.

**En palabras simples:** Un fold a la vez: entrena **RF** en k−1 particiones y mide la métrica `scoring` en la restante. En trading: Robusto al ruido; suele ganar el benchmark.

**Qué hace el código:** Un fold a la vez: entrena **RF** en k−1 particiones y mide la métrica `scoring` en la restante.

**Qué debes ver al ejecutar:** Línea `RF: media (std)` — anota la media.

**Qué implica para Bitcoin / trading:** Robusto al ruido; suele ganar el benchmark.

**Qué implica para el modelo ML:** 50 árboles en paralelo.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Línea `RF: media (std)` — anota la media.


In [ ]:
_found = False
for _code, _model in bench_list:
    if _code == 'RF':
        cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
        results.append(cv_results)
        names.append('RF')
        print('RF: %f (%f)' % (cv_results.mean(), cv_results.std()))
        _found = True
        break
if not _found:
    print('RF: no incluido en bench_list (omitido).')


### Paso C4.3-KNN — CV — KNN (solo modo completo)

**Objetivo:** Completar «CV — KNN (solo modo completo)» y verificar la salida antes de continuar.

**En palabras simples:** Evalúa **KNN** cuando `MODO_LIGERO = False`. En trading: Vecinos cercanos en RSI/EMA/etc.

**Qué hace el código:** Evalúa **KNN** cuando `MODO_LIGERO = False`.

**Qué debes ver al ejecutar:** En modo ligero verás mensaje de omitido.

**Qué implica para Bitcoin / trading:** Vecinos cercanos en RSI/EMA/etc.

**Qué implica para el modelo ML:** Clasificador por distancia en el espacio de features.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: En modo ligero verás mensaje de omitido.


In [ ]:
if MODO_LIGERO:
    print('KNN: omitido en MODO_LIGERO')
else:
    for _code, _model in bench_list:
        if _code == 'KNN':
            cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
            results.append(cv_results)
            names.append('KNN')
            print('KNN: %f (%f)' % (cv_results.mean(), cv_results.std()))
            break


### Paso C4.3-NB — CV — NB (solo modo completo)

**Objetivo:** Completar «CV — NB (solo modo completo)» y verificar la salida antes de continuar.

**En palabras simples:** Evalúa **NB** cuando `MODO_LIGERO = False`. En trading: Baseline probabilístico rápido.

**Qué hace el código:** Evalúa **NB** cuando `MODO_LIGERO = False`.

**Qué debes ver al ejecutar:** En modo ligero verás mensaje de omitido.

**Qué implica para Bitcoin / trading:** Baseline probabilístico rápido.

**Qué implica para el modelo ML:** Generativo con supuesto de independencia.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: En modo ligero verás mensaje de omitido.


In [ ]:
if MODO_LIGERO:
    print('NB: omitido en MODO_LIGERO')
else:
    for _code, _model in bench_list:
        if _code == 'NB':
            cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
            results.append(cv_results)
            names.append('NB')
            print('NB: %f (%f)' % (cv_results.mean(), cv_results.std()))
            break


### Paso C4.3-NN — CV — NN (solo modo completo)

**Objetivo:** Completar «CV — NN (solo modo completo)» y verificar la salida antes de continuar.

**En palabras simples:** Evalúa **NN** cuando `MODO_LIGERO = False`. En trading: Captura no linealidades.

**Qué hace el código:** Evalúa **NN** cuando `MODO_LIGERO = False`.

**Qué debes ver al ejecutar:** En modo ligero verás mensaje de omitido.

**Qué implica para Bitcoin / trading:** Captura no linealidades.

**Qué implica para el modelo ML:** MLP con capas ocultas.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: En modo ligero verás mensaje de omitido.


In [ ]:
if MODO_LIGERO:
    print('NN: omitido en MODO_LIGERO')
else:
    for _code, _model in bench_list:
        if _code == 'NN':
            cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
            results.append(cv_results)
            names.append('NN')
            print('NN: %f (%f)' % (cv_results.mean(), cv_results.std()))
            break


### Paso C4.3-AB — CV — AB (solo modo completo)

**Objetivo:** Completar «CV — AB (solo modo completo)» y verificar la salida antes de continuar.

**En palabras simples:** Evalúa **AB** cuando `MODO_LIGERO = False`. En trading: Secuencia de modelos correctoras.

**Qué hace el código:** Evalúa **AB** cuando `MODO_LIGERO = False`.

**Qué debes ver al ejecutar:** En modo ligero verás mensaje de omitido.

**Qué implica para Bitcoin / trading:** Secuencia de modelos correctoras.

**Qué implica para el modelo ML:** AdaBoost sobre clasificadores débiles.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: En modo ligero verás mensaje de omitido.


In [ ]:
if MODO_LIGERO:
    print('AB: omitido en MODO_LIGERO')
else:
    for _code, _model in bench_list:
        if _code == 'AB':
            cv_results = cross_val_score(_model, X_train, Y_train, cv=kfold, scoring=scoring)
            results.append(cv_results)
            names.append('AB')
            print('AB: %f (%f)' % (cv_results.mean(), cv_results.std()))
            break


### Paso C4.4 — Boxplot del benchmark

**Objetivo:** Completar «Boxplot del benchmark» y verificar la salida antes de continuar.

**En palabras simples:** Compara distribución de scores CV entre algoritmos.

**Qué hace el código:** Compara distribución de scores CV entre algoritmos.

**Qué debes ver al ejecutar:** Cajas estrechas = desempeño estable entre folds.

**Qué implica para el modelo ML:** Elige el ganador (suele ser RF/GBM) para el Grid Search.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Cajas estrechas = desempeño estable entre folds.


In [ ]:
fig = plt.figure()
fig.suptitle('Comparación de algoritmos')
ax = fig.add_subplot(111)
plt.boxplot(results, labels=names)
ax.set_ylabel(scoring)
fig.set_size_inches(15, 8)
plt.xticks(rotation=45)
plt.show()


**Interpretación (10 min discusión):** ¿Cuál gana en **media CV**? ¿La caja es estrecha (estable)? En clase, el ganador suele ser **RF o GBM**; anoten el nombre para el grid.

> **Actividad C1 (5 min):** cambien `scoring` a `'recall'` y vuelvan a correr **solo** la celda del benchmark. ¿Cambia el ranking? ¿Por qué importa para una estrategia long?


---
## Parte D

**Qué hacemos aquí (sin jerga):** Afinamos **Random Forest** (Grid Search), entrenamos el modelo final, lo evaluamos en **validation** y vemos **qué indicadores pesan más**.

**Al terminar la parte D, debes poder decir en voz alta:** Los mejores hiperparámetros, accuracy en validation, y qué significa cada celda de la matriz de confusión en dinero simulado.

**Pregunta guía:** ¿El Grid optimiza ganancias o solo accuracy? (respuesta: solo la métrica `scoring`, por defecto accuracy).


<a id='5'></a>
# 6. Ajuste fino y Grid Search

### Paso a paso — Grid Search (modelos)

1. **Elegir familia:** Random Forest (suele ganar el benchmark).
2. **Definir malla:** `n_estimators`, `max_depth`, `criterion` (Gini vs entropía).
3. **Escalar X_train** con `StandardScaler` (solo para el Grid Search).
4. **GridSearchCV** prueba cada combinación × cada fold → guarda `best_params_`.
5. **Interpretar heatmap:** celdas similares = zona robusta; pico aislado = cuidado.

### Paso a paso — hiperparámetros RF en lenguaje Bitcoin

- **`n_estimators`:** más árboles → menos varianza, más tiempo de cómputo.
- **`max_depth`:** árboles más profundos → captan micro-patrones del minuto (riesgo de sobreajuste).
- **`criterion`:** forma de medir “pureza” de cada split (Gini vs entropía); impacto suele ser menor que la profundidad.


### Paso D1 — Ajustar StandardScaler

**Objetivo:** Completar «Ajustar StandardScaler» y verificar la salida antes de continuar.

**En palabras simples:** Calcula media/desvío de cada feature en train y transforma X_train.

**Qué hace el código:** Calcula media/desvío de cada feature en train y transforma X_train.

**Qué debes ver al ejecutar:** Media ~0 en la primera feature tras escalar.

**Qué implica para el modelo ML:** Solo para Grid Search RF; el modelo final no usa escala.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Media ~0 en la primera feature tras escalar.


In [ ]:
scaler = StandardScaler().fit(X_train)
rescaledX = scaler.transform(X_train)
print('Media tras escalar (1ª feature, muestra):', rescaledX[:, 0].mean().round(4))


### Paso D2a — Hiperparámetro n_estimators

**Objetivo:** Completar «Hiperparámetro n_estimators» y verificar la salida antes de continuar.

**En palabras simples:** Lista de candidatos de número de árboles (según modo clase). En trading: Más árboles → señal más estable, más tiempo de cómputo.

**Qué hace el código:** Lista de candidatos de número de árboles (según modo clase).

**Qué debes ver al ejecutar:** Vector `n_estimators` impreso.

**Qué implica para Bitcoin / trading:** Más árboles → señal más estable, más tiempo de cómputo.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Vector `n_estimators` impreso.


In [ ]:
if MODO_LIGERO:
    n_estimators = [20, 50]
else:
    n_estimators = [20, 80]
print('n_estimators:', n_estimators)


### Paso D2b — Hiperparámetros max_depth y criterion

**Objetivo:** Completar «Hiperparámetros max_depth y criterion» y verificar la salida antes de continuar.

**En palabras simples:** Profundidad máxima de cada árbol y criterio de impureza (Gini/entropía).

**Qué hace el código:** Profundidad máxima de cada árbol y criterio de impureza (Gini/entropía).

**Qué debes ver al ejecutar:** Listas `max_depth` y `criterion`.

**Qué implica para el modelo ML:** max_depth bajo → menos sobreajuste al ruido minuto a minuto.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Listas `max_depth` y `criterion`.


In [ ]:
max_depth = [5, 10]
criterion = ['gini'] if MODO_LIGERO else ['gini', 'entropy']
print('max_depth:', max_depth, '| criterion:', criterion)


### Paso D2c — Armar param_grid

**Objetivo:** Completar «Armar param_grid» y verificar la salida antes de continuar.

**En palabras simples:** Diccionario con todas las combinaciones a probar.

**Qué hace el código:** Diccionario con todas las combinaciones a probar.

**Qué debes ver al ejecutar:** Número total de combinaciones.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Número total de combinaciones.


In [ ]:
param_grid = dict(n_estimators=n_estimators, max_depth=max_depth, criterion=criterion)
print('Combinaciones a probar:', len(n_estimators) * len(max_depth) * len(criterion))


### Paso D3 — GridSearchCV — entrenamiento

**Objetivo:** Completar «GridSearchCV — entrenamiento» y verificar la salida antes de continuar.

**En palabras simples:** Prueba cada combinación con CV sobre train escalado.

**Qué hace el código:** Prueba cada combinación con CV sobre train escalado.

**Qué debes ver al ejecutar:** `best_score_` y `best_params_` impresos.

**Qué implica para el modelo ML:** Elige hiperparámetros maximizando scoring en CV, no en validation.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: `best_score_` y `best_params_` impresos.


In [ ]:
model = RandomForestClassifier(n_jobs=-1, random_state=1)
kfold = KFold(n_splits=num_folds, shuffle=True, random_state=seed)
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring=scoring, cv=kfold, n_jobs=-1)
grid_result = grid.fit(rescaledX, Y_train)
print('Best CV score:', grid_result.best_score_.round(4))
print('Best params:', grid_result.best_params_)


### Paso D4 — Ranking de combinaciones

**Objetivo:** Completar «Ranking de combinaciones» y verificar la salida antes de continuar.

**En palabras simples:** Lista cada combinación con rank, media CV y desviación.

**Qué hace el código:** Lista cada combinación con rank, media CV y desviación.

**Qué debes ver al ejecutar:** Rank #1 coincide con `best_params_`.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Rank #1 coincide con `best_params_`.


In [ ]:
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
ranks = grid_result.cv_results_['rank_test_score']
for mean, stdev, param, rank in zip(means, stds, params, ranks):
    print('#%d %f (%f) with: %r' % (rank, mean, stdev, param))


### Paso D4b — Preparar tabla para heatmap

**Objetivo:** Completar «Preparar tabla para heatmap» y verificar la salida antes de continuar.

**En palabras simples:** Convierte resultados del grid en DataFrame.

**Qué hace el código:** Convierte resultados del grid en DataFrame.

**Qué debes ver al ejecutar:** Filas = una fila por combinación probada.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Filas = una fila por combinación probada.


In [ ]:
cv_df = pd.DataFrame(grid_result.cv_results_)
print('Filas en cv_df:', len(cv_df))


### Paso D4c — Heatmap Grid RF

**Objetivo:** Completar «Heatmap Grid RF» y verificar la salida antes de continuar.

**En palabras simples:** Mapa de calor: max_depth vs n_estimators (por cada criterion). En trading: Optimizamos accuracy CV, no PnL — el heatmap no garantiza ganancias.

**Qué hace el código:** Mapa de calor: max_depth vs n_estimators (por cada criterion).

**Qué debes ver al ejecutar:** Celdas similares = zona robusta de hiperparámetros.

**Qué implica para Bitcoin / trading:** Optimizamos accuracy CV, no PnL — el heatmap no garantiza ganancias.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Celdas similares = zona robusta de hiperparámetros.


In [ ]:
for crit in criterion:
    sub = cv_df[cv_df['param_criterion'] == crit]
    pivot = sub.pivot(index='param_max_depth', columns='param_n_estimators', values='mean_test_score')
    plt.figure(figsize=(5, 4))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis')
    plt.title('Grid RF — criterion=%s' % crit)
    plt.show()


<a id='6'></a>
# 7. Finalizar el modelo


<a id='6.1'></a>
## 7.1 Resultados en el conjunto de validación

### Paso a paso — evaluación final (ML + trading)

1. **Reentrenar** RF con `best_params_` sobre todo **X_train** (sin escalar).
2. **Predecir** en **X_validation** → vector `predictions`.
3. **Accuracy / reporte:** calidad global de clasificación.
4. **Matriz de confusión:** traducir TP/FP a “compras acertadas” vs “falsas alarmas”.
5. **Importancia de variables:** qué indicadores usa el bosque para imitar/mejorar la SMA.

Parámetros ganadores del grid; entrenamiento sobre **X_train sin escalar**.


### Paso D5 — Modelo final en train

**Objetivo:** Completar «Modelo final en train» y verificar la salida antes de continuar.

**En palabras simples:** Instancia RandomForest con `best_params_` y ajusta en X_train **sin escalar**.

**Qué hace el código:** Instancia RandomForest con `best_params_` y ajusta en X_train **sin escalar**.

**Qué debes ver al ejecutar:** Mensaje `Entrenado con:` y dict de parámetros.

**Qué implica para el modelo ML:** Escala solo en grid; no en el fit final.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Mensaje `Entrenado con:` y dict de parámetros.


In [ ]:
# Mejores hiperparámetros del Grid Search
bp = grid_result.best_params_
model = RandomForestClassifier(
    criterion=bp.get('criterion', 'gini'),
    n_estimators=bp.get('n_estimators', 80),
    max_depth=bp.get('max_depth', 10),
    n_jobs=-1,
    random_state=1,
)
model.fit(X_train, Y_train)
print('Entrenado con:', bp)


### Paso D6 — Predicciones ŷ en validation

**Objetivo:** Completar «Predicciones ŷ en validation» y verificar la salida antes de continuar.

**En palabras simples:** Aplica `predict` en X_validation. En trading: Cada 1 predicho ≈ minuto donde el modelo recomienda estar largo.

**Qué hace el código:** Aplica `predict` en X_validation.

**Qué debes ver al ejecutar:** Accuracy impresa vs Y_validation.

**Qué implica para Bitcoin / trading:** Cada 1 predicho ≈ minuto donde el modelo recomienda estar largo.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Accuracy impresa vs Y_validation.


In [ ]:
predictions = model.predict(X_validation)
print('Accuracy validation:', accuracy_score(Y_validation, predictions))


### Paso D7a — Matriz de confusión (números)

**Objetivo:** Completar «Matriz de confusión (números)» y verificar la salida antes de continuar.

**En palabras simples:** Cuenta TP, TN, FP, FN entre real y predicho. En trading: FP = falsa compra; FN = subida perdida.

**Qué hace el código:** Cuenta TP, TN, FP, FN entre real y predicho.

**Qué debes ver al ejecutar:** Matriz 2×2 impresa.

**Qué implica para Bitcoin / trading:** FP = falsa compra; FN = subida perdida.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Matriz 2×2 impresa.


In [ ]:
print(confusion_matrix(Y_validation, predictions))
print(classification_report(Y_validation, predictions))


### Paso D7b — Heatmap de confusión

**Objetivo:** Completar «Heatmap de confusión» y verificar la salida antes de continuar.

**En palabras simples:** Visualiza la misma matriz (más fácil de discutir en clase).

**Qué hace el código:** Visualiza la misma matriz (más fácil de discutir en clase).

**Qué debes ver al ejecutar:** Diagonal fuerte = buen acuerdo con etiqueta SMA.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Diagonal fuerte = buen acuerdo con etiqueta SMA.


In [ ]:
df_cm = pd.DataFrame(
    confusion_matrix(Y_validation, predictions),
    columns=np.unique(Y_validation),
    index=np.unique(Y_validation),
)
df_cm.index.name = 'Actual'
df_cm.columns.name = 'Predicted'
sns.heatmap(df_cm, cmap='Blues', annot=True, annot_kws={'size': 16})
plt.show()


**Matriz de confusión — traducción explícita (long-only)**

| | **Predicho 0** (no largo) | **Predicho 1** (largo) |
|---|---------------------------|-------------------------|
| **Real 0** | TN — acierto, evitamos compra mala | **FP — falsa compra** (el modelo compra cuando SMA decía no) |
| **Real 1** | **FN — perdimos subida** | TP — acierto, coincidimos con SMA alcista |

- **Accuracy alta** puede ocultar muchos **FP** (compras malas).
- **Precision** de clase 1 ≈ “cuando digo compra, ¿cuántas veces acierto?”.
- **Recall** de clase 1 ≈ “de todos los minutos buenos (SMA=1), ¿cuántos capturo?”.


<a id='6.2'></a>
## 7.2 Intuición de variables / importancia de features


### Paso D8a — Importancias del bosque

**Objetivo:** Completar «Importancias del bosque» y verificar la salida antes de continuar.

**En palabras simples:** Extrae `feature_importances_` del RF (suma 100% en el gráfico).

**Qué hace el código:** Extrae `feature_importances_` del RF (suma 100% en el gráfico).

**Qué debes ver al ejecutar:** Series ordenada por importancia.

**Qué implica para el modelo ML:** Impurity decrease — no es causalidad ni estabilidad temporal.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Series ordenada por importancia.


In [ ]:
Importance = pd.DataFrame({'Importance': model.feature_importances_ * 100}, index=X.columns)
Importance.sort_values('Importance', ascending=False).head(8)


### Paso D8b — Gráfico de barras

**Objetivo:** Completar «Gráfico de barras» y verificar la salida antes de continuar.

**En palabras simples:** Visualiza qué indicadores usa más el modelo. En trading: Si dominan RSI/ROC, el ML imita lógica momentum/tendencia.

**Qué hace el código:** Visualiza qué indicadores usa más el modelo.

**Qué debes ver al ejecutar:** Barras largas = splits más frecuentes en el bosque.

**Qué implica para Bitcoin / trading:** Si dominan RSI/ROC, el ML imita lógica momentum/tendencia.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Barras largas = splits más frecuentes en el bosque.


In [ ]:
Importance.sort_values('Importance', axis=0, ascending=True).plot(kind='barh', color='steelblue', figsize=(8, 6))
plt.xlabel('Importancia de variables (%)')
plt.tight_layout()
plt.show()


**Interpretación:** indicadores de **momentum** (RSI, ROC, estocástico) suelen aparecer arriba si la etiqueta SMA es tendencial. No implica causalidad ni estabilidad temporal: reentrena y compara importancias en distintas ventanas (walk-forward en ventanas distintas).


<a id='6.3'></a>
## 7.3 Guardar modelo para uso posterior


### Paso D9 — Persistir modelo

**Objetivo:** Completar «Persistir modelo» y verificar la salida antes de continuar.

**En palabras simples:** Guarda el RF entrenado con `pickle`.

**Qué hace el código:** Guarda el RF entrenado con `pickle`.

**Qué debes ver al ejecutar:** Archivo `finalized_model_bitcoin.sav` en el directorio de trabajo.

**Qué implica para el modelo ML:** En producción versionaría modelo + fecha + features usados.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Archivo `finalized_model_bitcoin.sav` en el directorio de trabajo.


In [ ]:
filename = 'finalized_model_bitcoin.sav'
dump(model, open(filename, 'wb'))
print('Modelo guardado:', filename)


---
## Parte E

**Qué hacemos aquí (sin jerga):** Convertimos predicciones 0/1 en **retornos minuto a minuto** y comparamos **estrategia ML vs regla SMA** (con un minuto de retraso).

**Al terminar la parte E, debes poder decir en voz alta:** Si la curva acumulada del ML queda arriba o abajo de la SMA, y qué falta (comisiones, split temporal) para confiar en dinero real.

**Pregunta guía:** ¿Por qué multiplicamos retorno × señal de t−1 y no de t?


<a id='7'></a>
# 8. Backtesting

### Paso a paso — simulación (aplicación Bitcoin)

1. **`Market Returns`** = $\frac{P_t - P_{t-1}}{P_{t-1}}$ (retorno del BTC en ese minuto).
2. **`signal_actual`** = etiqueta SMA (referencia / benchmark).
3. **`signal_pred`** = predicción del Random Forest.
4. **Posición con lag:** usamos `shift(1)` → la señal de $t-1$ multiplica el retorno de $t$ (evita usar la señal del mismo minuto del retorno en esta demo).
5. **`Strategy Returns`** = retorno de mercado × señal predicha (0 o 1) → simula estar largo solo cuando el modelo dice 1.
6. **`Actual Returns`** = mismo esquema con la señal SMA → comparar ML vs regla original.
7. **Gráfico acumulado:** suma simple de retornos (didáctico; en producción usar log-returns, comisiones y slippage).

Retorno × señal con **`.shift(1)`** (decisión en t−1, retorno en t).


### Paso E1a — Crear marco de backtest

**Objetivo:** Completar «Crear marco de backtest» y verificar la salida antes de continuar.

**En palabras simples:** DataFrame indexado como validation.

**Qué hace el código:** DataFrame indexado como validation.

**Qué debes ver al ejecutar:** Tabla vacía con índice correcto.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Tabla vacía con índice correcto.


In [ ]:
backtestdata = pd.DataFrame(index=X_validation.index)
print('Filas backtest:', len(backtestdata))


### Paso E1b — Señal predicha (ML)

**Objetivo:** Completar «Señal predicha (ML)» y verificar la salida antes de continuar.

**En palabras simples:** Columna `signal_pred` = salida del Random Forest. En trading: 1 = minuto donde el modelo estaría largo.

**Qué hace el código:** Columna `signal_pred` = salida del Random Forest.

**Qué debes ver al ejecutar:** Valores 0/1 en `signal_pred`.

**Qué implica para Bitcoin / trading:** 1 = minuto donde el modelo estaría largo.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Valores 0/1 en `signal_pred`.


In [ ]:
backtestdata['signal_pred'] = predictions
backtestdata['signal_pred'].value_counts()


### Paso E1c — Señal real (SMA)

**Objetivo:** Completar «Señal real (SMA)» y verificar la salida antes de continuar.

**En palabras simples:** Columna `signal_actual` = etiqueta SMA de referencia.

**Qué hace el código:** Columna `signal_actual` = etiqueta SMA de referencia.

**Qué debes ver al ejecutar:** Compara conteos con predicción.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Compara conteos con predicción.


In [ ]:
backtestdata['signal_actual'] = Y_validation
backtestdata[['signal_pred', 'signal_actual']].head(3)


### Paso E2 — Retorno de mercado

**Objetivo:** Completar «Retorno de mercado» y verificar la salida antes de continuar.

**En palabras simples:** `pct_change()` del Close en validation. En trading: Retorno buy-and-hold minuto a minuto sin filtro.

**Qué hace el código:** `pct_change()` del Close en validation.

**Qué debes ver al ejecutar:** Primer valor NaN (no hay t−1).

**Qué implica para Bitcoin / trading:** Retorno buy-and-hold minuto a minuto sin filtro.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Primer valor NaN (no hay t−1).


In [ ]:
backtestdata['Market Returns'] = X_validation['Close'].pct_change()
backtestdata[['Market Returns']].head(3)


### Paso E3 — Retorno estrategia SMA (benchmark)

**Objetivo:** Tener la curva de PnL de la regla SMA en el mismo hold-out que el modelo.

**En palabras simples:** Si la SMA decía ‘sí’ el minuto anterior, me quedo expuesto al movimiento de este minuto.

**Qué hace el código:** `Actual Returns = Market Returns * signal_actual.shift(1)`.

**Qué debes ver al ejecutar:** Columna `Actual Returns`; primera fila suele ser NaN por el lag.

**Qué implica para Bitcoin / trading:** **Interpretación:** solo gano el retorno del minuto t si **en t−1 la SMA decía estar largo (1).** Es la referencia SMA.

**Qué implica para el modelo ML:** No usamos ML aquí; es el baseline para comparar.

**Confirma por escrito (cuaderno o chat):** ¿Actual Returns y Market Returns son iguales en algún minuto? ¿Cuándo son cero?


In [ ]:
backtestdata['Actual Returns'] = backtestdata['Market Returns'] * backtestdata['signal_actual'].shift(1)
backtestdata[['Market Returns', 'signal_actual', 'Actual Returns']].head(5)


### Paso E4 — Retorno estrategia ML

**Objetivo:** Completar «Retorno estrategia ML» y verificar la salida antes de continuar.

**En palabras simples:** Market return × señal predicha en t−1. En trading: Compara PnL simple ML vs SMA en el mismo hold-out.

**Qué hace el código:** Market return × señal predicha en t−1.

**Qué debes ver al ejecutar:** Columna `Strategy Returns`.

**Qué implica para Bitcoin / trading:** Compara PnL simple ML vs SMA en el mismo hold-out.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Columna `Strategy Returns`.


In [ ]:
backtestdata['Strategy Returns'] = backtestdata['Market Returns'] * backtestdata['signal_pred'].shift(1)
backtestdata = backtestdata.reset_index()
backtestdata[['signal_pred', 'Strategy Returns']].head(5)


### Paso E5a — Retorno acumulado (números)

**Objetivo:** Completar «Retorno acumulado (números)» y verificar la salida antes de continuar.

**En palabras simples:** Suma simple de retornos minuto a minuto (didáctico). En trading: No incluye comisiones Bitstamp ni slippage.

**Qué hace el código:** Suma simple de retornos minuto a minuto (didáctico).

**Qué debes ver al ejecutar:** Totales acumulados impresos para ML vs SMA.

**Qué implica para Bitcoin / trading:** No incluye comisiones Bitstamp ni slippage.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Totales acumulados impresos para ML vs SMA.


In [ ]:
cum = backtestdata[['Strategy Returns', 'Actual Returns']].cumsum()
print('Acumulado final ML:', round(float(cum['Strategy Returns'].iloc[-1]), 4))
print('Acumulado final SMA:', round(float(cum['Actual Returns'].iloc[-1]), 4))


### Paso E5b — Histograma de caminos acumulados

**Objetivo:** Completar «Histograma de caminos acumulados» y verificar la salida antes de continuar.

**En palabras simples:** Distribución de retornos acumulados (exploración rápida).

**Qué hace el código:** Distribución de retornos acumulados (exploración rápida).

**Qué debes ver al ejecutar:** Forma del histograma — no es walk-forward.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Forma del histograma — no es walk-forward.


In [ ]:
backtestdata[['Strategy Returns', 'Actual Returns']].cumsum().hist()


### Paso E5c — Serie temporal acumulada

**Objetivo:** Completar «Serie temporal acumulada» y verificar la salida antes de continuar.

**En palabras simples:** Gráfico principal: ¿ML supera a SMA en validation?

**Qué hace el código:** Gráfico principal: ¿ML supera a SMA en validation?

**Qué debes ver al ejecutar:** Curva más alta al final = mejor en esta demo simplificada.

**Confirma por escrito (cuaderno o chat):** La salida coincide con: Curva más alta al final = mejor en esta demo simplificada.


In [ ]:
backtestdata[['Strategy Returns', 'Actual Returns']].cumsum().plot(figsize=(12, 5))
plt.title('Retornos acumulados (suma simple) — estrategia ML vs señal SMA')
plt.legend(['Estrategia ML', 'Señal SMA'])
plt.show()


### Conclusión

1. **Formulación:** convertir el objetivo de inversión en etiquetas y features es el primer paso; aquí la etiqueta sigue una regla SMA interpretable.
2. **Feature engineering:** indicadores de tendencia y momentum aportan información frente a usar solo precio crudo.
3. **Métricas:** accuracy es útil si las clases están balanceadas; para **long-only** revisa **precision** (falsas compras) y **recall** (subidas perdidas).
4. **Backtesting:** simula rentabilidad **antes** de arriesgar capital — sin comisiones, slippage ni tamaño de posición.

**Próximo paso:** CSV más grande en Drive, split temporal, comisiones y validación walk-forward.

> **Cierre:** ¿La estrategia ML supera a la señal SMA en el gráfico? ¿Qué falta para confiar en capital real?


<a id='8'></a>
# 9. Modo con más filas (opcional)

1. Pon **`MODO_LIGERO = False`**, usa un CSV más grande en **`RUTA_CSV`** y ejecuta **Runtime → Run all**.
2. Compara accuracy CV y backtest con el modo ligero.
3. Entrega sugerida: boxplot, `best_params_`, matriz de confusión e interpretación precision vs recall.
